## 0. Instalacao de Dependencias

Instala todas as bibliotecas usadas no notebook e o `poppler-utils` (dependencia de sistema do `unstructured[pdf]`). Apos executar as duas celulas abaixo, va em **Runtime > Restart session** e depois **Runtime > Run all**.


In [1]:
%pip install -q transformers pandas numpy sentencepiece accelerate pydantic langchain-community langchain-huggingface langchain-chroma langchain-text-splitters chromadb sentence-transformers "unstructured[md,pdf]" requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 33.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.2/542.2 kB 13.8 MB/s eta 0:00:00
 

In [2]:
!apt-get -qq update && apt-get -qq install -y poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up libpoppler118:amd64 (22.02.0-2ubuntu0.13) ...
Setting up poppler-

# Cybersecurity Incident Analyzer — Fase 1


---
## 1. Instalação de Dependências e Imports

Instalamos as bibliotecas necessárias para carregar modelos pré-treinados, manipular o dataset e avaliar os resultados.


In [3]:
import json
import re
import textwrap
from enum import Enum

import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel, Field, field_validator
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline,
)

device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"Dispositivo em uso: {device_name}")

if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")


Dispositivo em uso: GPU
GPU detectada: Tesla T4


In [4]:
def print_wrapped(text, width=100):
    for line in str(text).splitlines() or [""]:
        if line.strip():
            print(
                textwrap.fill(
                    line,
                    width=width,
                    break_long_words=False,
                    break_on_hyphens=False,
                )
            )
        else:
            print()


---
## 2. Dataset de Incidentes (Exemplos Fictícios)

O dataset abaixo contém **10 incidentes fictícios**. Cada exemplo inclui:

- o relato do incidente em inglês;
- comentários e explicações em português.

Nesta fase, os textos são usados para demonstrar sumarização e extração direta de entidades com NER.


In [5]:
incidents = [
    {
        "id": 1,
        "text": "On June 14th, multiple employees in the Finance department reported receiving an email claiming to be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract penalties'. One employee clicked the link and entered their corporate credentials on a fake Office 365 login page before realizing the site was illegitimate. IT Security was notified and the employee's account was immediately disabled pending investigation.",
        "expected_extraction": {
            "incident_type": "Phishing",
            "affected_asset": "Finance employee account",
            "observables": [
                "cfo-finance-corp.com",
                "Office 365"
            ],
            "impact": "An employee submitted corporate credentials to a fake Office 365 login page.",
            "response_actions": [
                "IT Security was notified",
                "The employee account was disabled pending investigation"
            ]
        }
    },
    {
        "id": 2,
        "text": "At 03:12 AM, the file server hosting the HR department's shared drive became unresponsive. Upon investigation, system administrators discovered that thousands of files had been encrypted and renamed with a '.lockedcorp' extension. A ransom note demanding 5 Bitcoin was found in every affected directory, threatening to leak employee personal data if payment was not made within 72 hours. Backups from the previous night appear unaffected and are being verified for integrity.",
        "expected_extraction": {
            "incident_type": "Ransomware",
            "affected_asset": "HR shared file server",
            "observables": [
                ".lockedcorp",
                "5 Bitcoin",
                "72 hours"
            ],
            "impact": "Thousands of files on the HR shared file server were encrypted.",
            "response_actions": [
                "Backups are being verified for integrity"
            ]
        }
    },
    {
        "id": 3,
        "text": "Endpoint detection software flagged unusual behavior on a marketing department laptop: a previously unseen executable was making repeated outbound connections to an IP address associated with known command-and-control infrastructure. The process was attempting to enumerate browser-stored credentials and capture keystrokes. The laptop was isolated from the network and a forensic image was taken for analysis.",
        "expected_extraction": {
            "incident_type": "Malware",
            "affected_asset": "Marketing laptop",
            "observables": [
                "command-and-control infrastructure",
                "browser-stored credentials",
                "keystrokes"
            ],
            "impact": "A malicious executable attempted to steal credentials and capture keystrokes from a laptop.",
            "response_actions": [
                "The laptop was isolated from the network",
                "A forensic image was taken for analysis"
            ]
        }
    },
    {
        "id": 4,
        "text": "Security monitoring detected that an employee's VPN credentials were used to log in simultaneously from two geographically distant locations — one from the corporate office in Sao Paulo and another from an IP address registered in Eastern Europe, less than ten minutes apart. The employee confirmed they had not traveled and had not shared their password. A password reset and credential rotation was initiated immediately, along with a review of recently accessed systems.",
        "expected_extraction": {
            "incident_type": "Credential Theft",
            "affected_asset": "Employee VPN account",
            "observables": [
                "Sao Paulo",
                "Eastern Europe"
            ],
            "impact": "The employee VPN account was accessed from impossible-travel locations within minutes.",
            "response_actions": [
                "A password reset was initiated",
                "Credential rotation was initiated",
                "Recently accessed systems were reviewed"
            ]
        }
    },
    {
        "id": 5,
        "text": "A system administrator with privileged access to the customer database was found to have exported a large volume of customer records to a personal USB drive shortly after receiving a negative performance review. The export occurred outside of normal working hours and was not associated with any approved business task. Data Loss Prevention (DLP) tooling flagged the transfer for review.",
        "expected_extraction": {
            "incident_type": "Insider Threat",
            "affected_asset": "Customer database",
            "observables": [
                "personal USB drive",
                "Data Loss Prevention"
            ],
            "impact": "A privileged administrator exported customer records to a personal USB drive.",
            "response_actions": [
                "The transfer was flagged for review by DLP tooling"
            ]
        }
    },
    {
        "id": 6,
        "text": "The authentication logs for the company's customer-facing web portal show over 40,000 failed login attempts within a two-hour window, originating from a distributed set of IP addresses and targeting a list of common username patterns. The pattern is consistent with an automated credential-stuffing or brute-force attack attempting to compromise customer accounts using leaked password lists.",
        "expected_extraction": {
            "incident_type": "Brute Force Attack",
            "affected_asset": "Customer-facing web portal",
            "observables": [
                "40,000 failed login attempts",
                "distributed IP addresses",
                "leaked password lists"
            ],
            "impact": "The customer-facing web portal was targeted by a large credential-stuffing campaign.",
            "response_actions": []
        }
    },
    {
        "id": 7,
        "text": "The company's primary e-commerce website became completely unreachable for approximately 45 minutes during a major promotional sale. Network monitoring showed an abnormal surge of incoming traffic, estimated at over 200 times normal volume, originating from thousands of distinct IP addresses worldwide and overwhelming the load balancers. The traffic pattern strongly suggests a coordinated denial-of-service attack timed to coincide with peak sales traffic.",
        "expected_extraction": {
            "incident_type": "DDoS",
            "affected_asset": "Primary e-commerce website",
            "observables": [
                "45 minutes",
                "200 times normal volume",
                "thousands of distinct IP addresses"
            ],
            "impact": "The primary e-commerce website became unreachable during a traffic flood.",
            "response_actions": []
        }
    },
    {
        "id": 8,
        "text": "A help desk technician received a phone call from someone claiming to be a senior executive who had 'forgotten' their password and urgently needed it reset to access an important client presentation. The caller was able to provide the executive's employee ID and date of birth, likely gathered from a previous data breach, and pressured the technician to bypass standard identity verification procedures.",
        "expected_extraction": {
            "incident_type": "Phishing",
            "affected_asset": "Executive account",
            "observables": [
                "employee ID",
                "date of birth",
                "phone call"
            ],
            "impact": "A caller used social engineering to pressure the help desk into resetting an executive account password.",
            "response_actions": []
        }
    },
    {
        "id": 9,
        "text": "Anti-virus software on a developer's workstation quarantined a file disguised as a legitimate npm package dependency that had been recently added to a project. Analysis revealed the package contained obfuscated code designed to harvest environment variables, including API keys and cloud credentials, and transmit them to an external server upon installation.",
        "expected_extraction": {
            "incident_type": "Malware",
            "affected_asset": "Developer workstation",
            "observables": [
                "npm package dependency",
                "API keys",
                "cloud credentials"
            ],
            "impact": "A malicious package attempted to harvest API keys and cloud credentials from a developer workstation.",
            "response_actions": [
                "Anti-virus software quarantined the file"
            ]
        }
    },
    {
        "id": 10,
        "text": "Several customer support agents reported that their session tokens appeared to be reused from unfamiliar devices shortly after they had logged into the support ticketing platform from a coffee shop's public Wi-Fi network. Investigation suggests the sessions may have been intercepted via an unsecured network, allowing an attacker to hijack active sessions without needing the original password.",
        "expected_extraction": {
            "incident_type": "Credential Theft",
            "affected_asset": "Support ticketing platform sessions",
            "observables": [
                "session tokens",
                "public Wi-Fi",
                "unfamiliar devices"
            ],
            "impact": "Active support platform sessions may have been hijacked after token interception on public Wi-Fi.",
            "response_actions": []
        }
    }
]

df_incidents = pd.DataFrame(
    {
        "id": [incident["id"] for incident in incidents],
        "incident_type": [incident["expected_extraction"]["incident_type"] for incident in incidents],
        "affected_asset": [incident["expected_extraction"]["affected_asset"] for incident in incidents],
    }
)

print(f"Total de incidentes no dataset: {len(df_incidents)}")
df_incidents


Total de incidentes no dataset: 10


,id,incident_type,affected_asset
0,1,Phishing,Finance employee account
1,2,Ransomware,HR shared file server
2,3,Malware,Marketing laptop
3,4,Credential Theft,Employee VPN account
4,5,Insider Threat,Customer database
5,6,Brute Force Attack,Customer-facing web portal
6,7,DDoS,Primary e-commerce website
7,8,Phishing,Executive account
8,9,Malware,Developer workstation
9,10,Credential Theft,Support ticketing platform sessions


---
## 3. Tarefa 1 — Sumarização de Incidentes

Relatos de incidentes em ambientes reais frequentemente chegam em formato longo e narrativo. A sumarização automática ajuda o analista a entender o núcleo do caso sem ler o texto inteiro antes da triagem.

Utilizamos o modelo **`facebook/bart-large-cnn`**, um modelo encoder-decoder pré-treinado para tarefas de sumarização em inglês.


In [6]:
summarization_model_name = "facebook/bart-large-cnn"
summarization_tokenizer = AutoTokenizer.from_pretrained(summarization_model_name)
summarization_model = AutoModelForSeq2SeqLM.from_pretrained(summarization_model_name)
summarization_model.eval()

if torch.cuda.is_available():
    summarization_model = summarization_model.to("cuda")


def summarize_text(text, max_length=60, min_length=15):
    inputs = summarization_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    model_device = next(summarization_model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        summary_ids = summarization_model.generate(
            **inputs,
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            early_stopping=True,
        )

    return summarization_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [7]:
example_incident = incidents[0]
summary = summarize_text(example_incident["text"])

print("INCIDENTE ORIGINAL:")
print_wrapped(example_incident["text"])
print()
print("SUMARIZACAO GERADA:")
print_wrapped(summary)


INCIDENTE ORIGINAL:
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a
spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract
penalties'. One employee clicked the link and entered their corporate credentials on a fake Office
365 login page before realizing the site was illegitimate. IT Security was notified and the
employee's account was immediately disabled pending investigation.

SUMARIZACAO GERADA:
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO. The email contained a spoofed domain (cfo-finance-corp.com) and urged recipients to
act within one hour to avoid 'contract penalties' One


In [8]:
summaries = []
for incident in incidents:
    summaries.append(
        summarize_text(
            incident["text"],
            max_length=50,
            min_length=10,
        )
    )

summary_df = pd.DataFrame(
    {
        "id": [incident["id"] for incident in incidents],
        "incident_type": [incident["expected_extraction"]["incident_type"] for incident in incidents],
        "summary": summaries,
    }
)

summary_df


,id,incident_type,summary
0,1,Phishing,"On June 14th, multiple employees in the Financ..."
1,2,Ransomware,Thousands of files were encrypted and renamed ...
2,3,Malware,A marketing department laptop was targeted by ...
3,4,Credential Theft,Security monitoring detected that an employee'...
4,5,Insider Threat,A system administrator exported a large volume...
5,6,Brute Force Attack,The authentication logs for the company's cust...
6,7,DDoS,The company's primary e-commerce website becam...
7,8,Phishing,A help desk technician received a phone call f...
8,9,Malware,Anti-virus software quarantined a file disguis...
9,10,Credential Theft,Several customer support agents reported that ...


---
## 4. Tarefa 2 — Extração com NER

Nesta etapa, usamos um modelo **NER pré-treinado** para identificar entidades diretamente no texto do incidente.

Aqui a saída é mantida de forma simples: mostramos **o retorno do próprio pipeline de NER**, sem regras adicionais para montar campos estruturados.


In [9]:
ner_model_name = "dslim/bert-base-NER"
ner_extractor = pipeline(
    "token-classification",
    model=ner_model_name,
    aggregation_strategy="simple",
    device=device,
)


def extract_named_entities(text):
    return ner_extractor(text)


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
for incident_id in [1, 4, 9]:
    incident = next(item for item in incidents if item["id"] == incident_id)
    extracted_entities = extract_named_entities(incident["text"])

    print(f"INCIDENTE #{incident_id}")
    print_wrapped(incident["text"])
    print()
    print("SAIDA DO NER:")
    for entity in extracted_entities:
        print(entity)
    print("-" * 80)


INCIDENTE #1
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a
spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract
penalties'. One employee clicked the link and entered their corporate credentials on a fake Office
365 login page before realizing the site was illegitimate. IT Security was notified and the
employee's account was immediately disabled pending investigation.

SAIDA DO NER:
{'entity_group': 'ORG', 'score': np.float32(0.86613643), 'word': 'Finance department', 'start': 40, 'end': 58}
{'entity_group': 'ORG', 'score': np.float32(0.7444848), 'word': 'CF', 'start': 111, 'end': 113}
{'entity_group': 'MISC', 'score': np.float32(0.9689775), 'word': 'Office 365', 'start': 389, 'end': 399}
{'entity_group': 'ORG', 'score': np.float32(0.99692655), 'word': 'IT Security', 'start': 455, 'end': 466}

In [11]:
ner_results = []
for incident in incidents:
    ner_results.append(
        {
            "id": incident["id"],
            "ner_entities": extract_named_entities(incident["text"]),
        }
    )

ner_results_df = pd.DataFrame(ner_results)
ner_results_df


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,ner_entities
0,1,"[{'entity_group': 'ORG', 'score': 0.86613643, ..."
1,2,"[{'entity_group': 'ORG', 'score': 0.51365846, ..."
2,3,[]
3,4,"[{'entity_group': 'ORG', 'score': 0.45450038, ..."
4,5,"[{'entity_group': 'MISC', 'score': 0.94700605,..."
5,6,[]
6,7,[]
7,8,[]
8,9,[]
9,10,"[{'entity_group': 'MISC', 'score': 0.97701454,..."


---
## 5. Leitura da Saída do NER

Cada item retornado pelo pipeline contém, em geral:

- `entity_group`: o tipo agregado da entidade reconhecida;
- `score`: a confiança do modelo;
- `word`: o trecho textual identificado;
- `start` e `end`: a posição da entidade no texto original.

Esse formato já é suficiente para inspecionar rapidamente quais pessoas, organizações, localidades e outros trechos relevantes o modelo reconheceu em cada relato.


In [12]:
entity_group_counts = {}

for row in ner_results:
    for entity in row["ner_entities"]:
        entity_group = entity["entity_group"]
        entity_group_counts[entity_group] = entity_group_counts.get(entity_group, 0) + 1

entity_group_df = pd.DataFrame(
    {
        "entity_group": list(entity_group_counts.keys()),
        "count": list(entity_group_counts.values()),
    }
).sort_values("count", ascending=False)

entity_group_df


,entity_group,count
1,MISC,8
0,ORG,6
2,LOC,2


---
## 6. Tokenização e Arquiteturas Relevantes

### Encoder-decoder

Modelos encoder-decoder são apropriados para tarefas de **transformação de sequência**, em que uma entrada textual precisa ser convertida em outra saída textual mais compacta ou estruturada. Neste notebook, usamos essa família para a **sumarização** com `facebook/bart-large-cnn`.

### Token classification / encoder-only

Modelos de token classification rotulam trechos do texto diretamente na entrada. Isso é útil para **Named Entity Recognition (NER)** e extração direta de entidades. Neste notebook, usamos essa família com `dslim/bert-base-NER`.

### Resumo comparativo

| Componente | Tipo de modelo | Papel na Fase 1 |
|---|---|---|
| `facebook/bart-large-cnn` | Encoder-decoder | Sumarizar relatos de incidentes |
| `dslim/bert-base-NER` | Token classification | Retornar entidades reconhecidas no texto |


---
## 7. Limitações Observadas Nesta Fase

Mesmo com o escopo mais enxuto desta fase, ainda existem limitações importantes.

### Cobertura de entidades

Um modelo NER genérico não foi treinado especificamente para o domínio de incidentes de segurança. Por isso, vários observáveis relevantes, como extensões maliciosas, nomes de técnicas ou artefatos específicos, podem não ser reconhecidos como entidades.

### Saída bruta do modelo

A saída do pipeline mostra somente as entidades detectadas pelo modelo, com seus rótulos e scores. Ela não organiza automaticamente essas entidades em um schema operacional de incidente.

### Conhecimento pré-treinado estático

Todos os modelos usados aqui dependem apenas do conhecimento aprendido no pré-treinamento. Eles não conhecem:

- playbooks internos da organização;
- inventário real de ativos críticos;
- incidentes passados da empresa;
- ameaças e indicadores publicados após o corte de treinamento.


# Cybersecurity Incident Analyzer — Fase 2

## Prompt Engineering e Saídas Controladas

Este bloco continua o mesmo dataset da Fase 1 e aplica três técnicas de prompt engineering à triagem automatizada de incidentes de cibersegurança para times SOC.

| # | Técnica | Ideia central |
|---|---------|---------------|
| 1 | **Role + Context + Task + Output Format (RCTOF)** | Prompt totalmente especificado com role priming e esquema JSON explícito |
| 2 | **Guided Chain of Thought (GCoT)** | O prompt guia um raciocínio em etapas antes da saída final |
| 3 | **Meta Prompting** | O LLM gera o prompt ideal; esse prompt conduz a análise final |

As três técnicas usam o **mesmo relato de incidente** e miram o **mesmo esquema de saída em JSON**.


---
## 1. Setup e Imports

In [13]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 768

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
elif torch.cuda.is_available():
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MODEL_DEVICE = next(model.parameters()).device

print(f"Model  : {MODEL_ID}")
print(f"Device : {MODEL_DEVICE}")
print(f"DType  : {MODEL_DTYPE}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model  : Qwen/Qwen2.5-1.5B-Instruct
Device : cuda:0
DType  : torch.bfloat16


---
## 2. Definição do Incidente

Os incidentes são reutilizados diretamente do array `incidents` definido no início deste notebook. Um item é selecionado com `INCIDENT_INDEX` e reaproveitado nas três técnicas para permitir comparação direta.

> **Dataset base:** os 10 incidentes fictícios usados em todo o projeto. A seleção inicial abaixo usa o incidente de índice `0`.


In [14]:
INCIDENT_INDEX = 0
selected_incident = incidents[INCIDENT_INDEX]
INCIDENT_TEXT = selected_incident["text"]

print(f"Loaded {len(incidents)} incidents from the in-memory incidents array")
print(f"Using incident #{selected_incident['id']}")
print(f"Incident length: {len(INCIDENT_TEXT)} characters")
print()
print("─" * 70)
print_wrapped(INCIDENT_TEXT, width=70)
print("─" * 70)


Loaded 10 incidents from the in-memory incidents array
Using incident #1
Incident length: 554 characters

──────────────────────────────────────────────────────────────────────
On June 14th, multiple employees in the Finance department reported
receiving an email claiming to be from the CFO, requesting urgent wire
transfers to a new vendor account. The email contained a spoofed
domain (cfo-finance-corp.com) and urged recipients to act within one
hour to avoid 'contract penalties'. One employee clicked the link and
entered their corporate credentials on a fake Office 365 login page
before realizing the site was illegitimate. IT Security was notified
and the employee's account was immediately disabled pending
investigation.
──────────────────────────────────────────────────────────────────────


---
## 3. Esquema de Saída (Pydantic v2)

As três técnicas precisam produzir saídas compatíveis com este esquema fixo.

O Pydantic v2 valida:
- `severity` pertence a `low | medium | high | critical` (Enum, rejeitando strings inválidas)
- `confidence` é um float `0.0 ≤ x ≤ 1.0`
- `recommended_actions` é uma lista não vazia de strings não vazias
- `summary` tem no mínimo 10 caracteres

In [15]:
class SeverityLevel(str, Enum):
    low = "low"
    medium = "medium"
    high = "high"
    critical = "critical"


class IncidentAnalysis(BaseModel):
    incident_type: str = Field(..., description="Type of cybersecurity incident")
    severity: SeverityLevel = Field(..., description="Severity: low/medium/high/critical")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence score 0.0-1.0")
    summary: str = Field(..., min_length=10, description="2-3 sentence incident summary")
    recommended_actions: list[str] = Field(
        ..., min_length=1, description="Prioritized SOC response actions"
    )

    @field_validator("severity", mode="before")
    @classmethod
    def normalize_severity(cls, v):
        return v.strip().lower() if isinstance(v, str) else v

    @field_validator("confidence")
    @classmethod
    def round_confidence(cls, v: float) -> float:
        return round(v, 3)

    @field_validator("recommended_actions")
    @classmethod
    def actions_not_empty(cls, v: list[str]) -> list[str]:
        if not all(isinstance(a, str) and a.strip() for a in v):
            raise ValueError("All actions must be non-empty strings")
        return v


print("Schema: IncidentAnalysis")
print()
schema = IncidentAnalysis.model_json_schema()
print(json.dumps(schema, indent=2))

Schema: IncidentAnalysis

{
  "$defs": {
    "SeverityLevel": {
      "enum": [
        "low",
        "medium",
        "high",
        "critical"
      ],
      "title": "SeverityLevel",
      "type": "string"
    }
  },
  "properties": {
    "incident_type": {
      "description": "Type of cybersecurity incident",
      "title": "Incident Type",
      "type": "string"
    },
    "severity": {
      "$ref": "#/$defs/SeverityLevel",
      "description": "Severity: low/medium/high/critical"
    },
    "confidence": {
      "description": "Confidence score 0.0-1.0",
      "maximum": 1.0,
      "minimum": 0.0,
      "title": "Confidence",
      "type": "number"
    },
    "summary": {
      "description": "2-3 sentence incident summary",
      "minLength": 10,
      "title": "Summary",
      "type": "string"
    },
    "recommended_actions": {
      "description": "Prioritized SOC response actions",
      "items": {
        "type": "string"
      },
      "minItems": 1,
      "title": "R

---
## 4. Utilitários de Geração, Parsing e Validação

Pipeline completo: `prompt de chat → geração com modelo open source → parse de JSON → validação com Pydantic`

### Tratamento de erros

| Modo de falha | Tratamento |
|---|---|
| JSON envolto em markdown fences | Remoção via regex |
| `json.loads()` falha | `ValueError` com contexto |
| Validação do Pydantic falha | `ValidationError` com detalhe por campo |

In [16]:
def generate_response_text(system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS):
    # Monta a conversa no formato esperado pelo modelo e gera apenas a continuação.
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt_text, return_tensors="pt")
    model_inputs = {k: v.to(MODEL_DEVICE) for k, v in model_inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_tokens = model_inputs["input_ids"].shape[-1]
    completion_tokens = generated_ids.shape[-1] - prompt_tokens
    completion_ids = generated_ids[0][prompt_tokens:]
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    return completion_text, prompt_tokens, completion_tokens


def parse_json_response(text):
    # Extrai e interpreta um objeto JSON a partir da saída textual bruta.
    text = text.strip()
    decoder = json.JSONDecoder()

    # 1. Tentativa direta
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. Remoção de cercas de markdown
    for pattern in [r"```json\s*(.*?)\s*```", r"```\s*(.*?)\s*```"]:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                continue

    # 3. Busca incremental pelo primeiro objeto JSON bem-formado.
    for start, char in enumerate(text):
        if char != "{":
            continue
        try:
            candidate, _ = decoder.raw_decode(text[start:])
            if isinstance(candidate, dict):
                return candidate
        except json.JSONDecodeError:
            continue

    raise ValueError(f"No valid JSON found in response:\n{text[:500]}")


def validate_analysis(data):
    # Valida o dicionário contra o schema IncidentAnalysis.
    if isinstance(data, dict) and isinstance(data.get("severity"), str):
        data = {**data, "severity": data["severity"].strip().lower()}
    return IncidentAnalysis(**data)


def repair_analysis_output(raw_text, max_new_tokens=384):
    # Reescreve uma resposta textual livre em JSON estrito quando o parse inicial falha.
    repair_system = (
        "You convert cybersecurity incident analyses into strict JSON. "
        "Return only a valid JSON object with no markdown, headings, or commentary."
    )
    repair_user = (
        "Convert the analysis below into a single valid JSON object.\n\n"
        "Rules:\n"
        "- Return ONLY JSON.\n"
        "- Use exactly this schema:\n"
        + _JSON_SCHEMA
        + "\n\n"
        "- severity must be one of: low, medium, high, critical (lowercase).\n"
        "- confidence must be a float between 0.0 and 1.0.\n"
        "- recommended_actions must be a non-empty list of concise strings.\n"
        "- If the source text is ambiguous, choose the most likely single incident_type and lower confidence instead of inventing facts.\n\n"
        "Source analysis:\n"
        + raw_text[:4000]
    )
    return generate_response_text(repair_system, repair_user, max_new_tokens=max_new_tokens)


def run_analysis(system_prompt, user_prompt, label="Analysis", max_new_tokens=MAX_NEW_TOKENS):
    # Executa a geração, interpreta o JSON e valida a resposta final.
    print(f"[{label}] Sending request...")

    text, input_tokens, output_tokens = generate_response_text(
        system_prompt,
        user_prompt,
        max_new_tokens=max_new_tokens,
    )

    raw_data = None
    analysis = None
    parse_ok = False
    validation_ok = False
    error = None
    repair_attempted = False
    repair_succeeded = False
    repair_text = None
    repair_input_tokens = 0
    repair_output_tokens = 0

    try:
        raw_data = parse_json_response(text)
        parse_ok = True
    except ValueError as e:
        error = f"ParseError: {e}"

    if parse_ok:
        try:
            analysis = validate_analysis(raw_data)
            validation_ok = True
        except Exception as e:
            error = f"ValidationError: {e}"

    if not validation_ok and text:
        print(f"[{label}] Attempting JSON repair...")
        repair_attempted = True
        repair_text, repair_input_tokens, repair_output_tokens = repair_analysis_output(text)
        try:
            raw_data = parse_json_response(repair_text)
            parse_ok = True
            analysis = validate_analysis(raw_data)
            validation_ok = True
            error = None
            repair_succeeded = True
        except Exception as e:
            error = f"RepairFailed: {e}"

    status = "OK" if validation_ok else f"FAILED — {error}"
    print_wrapped(f"[{label}] Status  : {status}")
    print(f"[{label}] Tokens  : {input_tokens} in / {output_tokens} out")
    if repair_attempted:
        print(f"[{label}] Repair  : {repair_input_tokens} in / {repair_output_tokens} out")

    return {
        "label": label,
        "text": text,
        "repair_text": repair_text,
        "raw_data": raw_data,
        "analysis": analysis,
        "parse_ok": parse_ok,
        "validation_ok": validation_ok,
        "error": error,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "repair_attempted": repair_attempted,
        "repair_succeeded": repair_succeeded,
        "repair_input_tokens": repair_input_tokens,
        "repair_output_tokens": repair_output_tokens,
    }


def display_result(result):
    # Exibe de forma legível uma resposta validada.
    sep = "=" * 65
    print(sep)
    print(f"  {result['label']}")
    print(sep)

    if not result["validation_ok"]:
        print_wrapped(f"  ERROR: {result['error']}")
        if result["text"]:
            print("\n  Raw text (first 800 chars):")
            print_wrapped(result['text'][:800])
        if result.get("repair_text"):
            print("\n  Repair text (first 800 chars):")
            print_wrapped(result["repair_text"][:800])
        return

    analysis = result["analysis"]
    print(f"  Incident Type  : {analysis.incident_type}")
    print(f"  Severity       : {analysis.severity.value.upper()}")
    print(f"  Confidence     : {analysis.confidence:.3f}")
    print_wrapped(f"  Summary        : {analysis.summary}")
    print(f"  Actions ({len(analysis.recommended_actions)}):")
    for i, action in enumerate(analysis.recommended_actions, 1):
        print(f"    {i}. {action}")
    if result.get("repair_succeeded"):
        print("  Repair Used    : YES")
    print(f"  Tokens         : {result['input_tokens']} in / {result['output_tokens']} out")


print("Utilities loaded.")

Utilities loaded.


---
## 5. Técnica 1 — Role + Context + Task + Output Format (RCTOF)

### Racional de desenho

O padrão **RCTOF** organiza o prompt em quatro elementos explícitos:

| Elemento | Conteúdo | Propósito |
|----------|----------|------------|
| **Role** | Analista sênior de cibersegurança | Priming de domínio, alinhando o comportamento do modelo |
| **Context** | Cenário de triagem SOC | Enquadramento situacional, reduzindo ambiguidades |
| **Task** | Classificar, avaliar, resumir e recomendar | Escopo explícito da instrução |
| **Output Format** | Esquema JSON exato dentro do prompt | Restrição estrutural da saída |

### Comportamento esperado
O modelo produz JSON diretamente, sem explicações extras. Há apenas uma geração, com baixo overhead.

In [17]:
T1_SYSTEM = (
    "You are a senior cybersecurity analyst specializing in incident response and threat intelligence. "
    "You have extensive experience with SOC operations, incident classification, and triage procedures "
    "across large enterprise environments."
)

_JSON_SCHEMA = (
    '{\n'
    '  "incident_type": "<string: attack category>",\n'
    '  "severity": "<one of: low | medium | high | critical>",\n'
    '  "confidence": <float: 0.0 to 1.0>,\n'
    '  "summary": "<string: 2-3 sentence description of what happened>",\n'
    '  "recommended_actions": [\n'
    '    "<string: first prioritized action>",\n'
    '    "<string: second prioritized action>"\n'
    '  ]\n'
    '}'
)

T1_USER = (
    "## Context\n"
    "Your organization's SOC has received a cybersecurity incident report requiring immediate triage.\n\n"
    "## Task\n"
    "Analyze the incident report below. Identify the attack type, assess severity, estimate your "
    "confidence level, write a concise summary, and list prioritized response actions.\n\n"
    "## Incident Report\n"
    + INCIDENT_TEXT
    + "\n\n"
    "## Output Format\n"
    "Respond with ONLY a valid JSON object — no explanation, no markdown fences, no preamble.\n"
    "Use exactly this schema:\n"
    + _JSON_SCHEMA
)

print(f"System prompt : {len(T1_SYSTEM)} chars")
print(f"User prompt   : {len(T1_USER)} chars")
print()
print("─" * 65)
print_wrapped(T1_USER[:650], width=65)
print("... [truncated]")

System prompt : 233 chars
User prompt   : 1327 chars

─────────────────────────────────────────────────────────────────
## Context
Your organization's SOC has received a cybersecurity incident
report requiring immediate triage.

## Task
Analyze the incident report below. Identify the attack type,
assess severity, estimate your confidence level, write a concise
summary, and list prioritized response actions.

## Incident Report
On June 14th, multiple employees in the Finance department
reported receiving an email claiming to be from the CFO,
requesting urgent wire transfers to a new vendor account. The
email contained a spoofed domain (cfo-finance-corp.com) and urged
recipients to act within one hour to avoid 'contract penalties'.
One employee clicked the link
... [truncated]


In [18]:
result_t1 = run_analysis(T1_SYSTEM, T1_USER, label="T1: RCTOF")
print()
display_result(result_t1)

[T1: RCTOF] Sending request...
[T1: RCTOF] Status  : OK
[T1: RCTOF] Tokens  : 327 in / 94 out

  T1: RCTOF
  Incident Type  : Phishing
  Severity       : HIGH
  Confidence     : 0.900
  Summary        : Employees in the Finance department were tricked into clicking a phishing email
that requested urgent wire transfers to a fake vendor account.
  Actions (2):
    1. Enable multi-factor authentication for all financial transactions.
    2. Improve training programs to educate employees about common phishing tactics.
  Tokens         : 327 in / 94 out


---
## 6. Técnica 2 — Guided Chain of Thought (GCoT)

### Racional de desenho

**Chain of Thought** orienta o modelo a organizar o raciocínio em etapas intermediárias antes de chegar à conclusão. Aqui, essas etapas são **explicitamente guiadas** e cada uma aponta para um campo específico da saída.

| Etapa | Alvo do raciocínio | Campo de saída |
|------|---------------------|----------------|
| 1 | Identificação do vetor de ataque | ajuda `incident_type` |
| 2 | Classificação do incidente | `incident_type` |
| 3 | Avaliação de severidade (tríade CIA) | `severity` |
| 4 | Calibração de confiança baseada em evidências | `confidence` |
| 5 | Resumo do incidente | `summary` |
| 6 | Ações priorizadas de resposta | `recommended_actions` |

### Comportamento esperado
Mesmo sem expor o raciocínio intermediário, a estrutura guiada tende a melhorar a calibração e a consistência da saída, ao custo de prompts mais longos.

In [19]:
T2_SYSTEM = (
    "You are a senior cybersecurity analyst with deep expertise in threat analysis and incident response. "
    "You approach every incident with systematic, evidence-based reasoning — examining attack vectors, "
    "blast radius, and response urgency before reaching conclusions."
)

_COT_STEPS = (
    "**Step 1 — Identify the attack vector:**\n"
    "What mechanism was used? Look for: delivery method, exploitation technique, initial access path.\n\n"
    "**Step 2 — Classify the incident type:**\n"
    "Based on the vector and observed behavior, assign an incident type. Consider: Ransomware, "
    "Phishing, Malware, DDoS, Insider Threat, Credential Theft, Brute Force, Supply Chain, or other.\n\n"
    "**Step 3 — Assess severity (low / medium / high / critical):**\n"
    "Evaluate business impact across:\n"
    "- Confidentiality: Was sensitive data exposed or at risk?\n"
    "- Integrity: Were systems or data modified?\n"
    "- Availability: Were services disrupted?\n"
    "- Scope: How many users/systems affected?\n"
    "- Urgency: Is the threat ongoing?\n\n"
    "**Step 4 — Estimate confidence (0.0–1.0):**\n"
    "How certain are you given available evidence?\n"
    "- High (>0.85): clear indicators, multiple corroborating signals\n"
    "- Medium (0.65–0.85): some ambiguity or missing evidence\n"
    "- Low (<0.65): significant uncertainty or contradictory signals\n\n"
    "**Step 5 — Write a concise incident summary (2-3 sentences):**\n"
    "Cover: what happened, who/what was affected, key indicators observed.\n\n"
    "**Step 6 — List prioritized response actions:**\n"
    "Specific, actionable steps for the SOC team, ordered by priority.\n\n"
)

_FINAL_JSON = (
    '{\n'
    '  "incident_type": "<string>",\n'
    '  "severity": "<low|medium|high|critical>",\n'
    '  "confidence": <float 0.0-1.0>,\n'
    '  "summary": "<string>",\n'
    '  "recommended_actions": ["<string>", ...]\n'
    '}'
)

T2_USER = (
    "Analyze this cybersecurity incident using the structured reasoning steps below.\n"
    "Work through each step carefully before producing the final JSON output.\n\n"
    "**Incident Report:**\n"
    + INCIDENT_TEXT
    + "\n\n---\n\n"
    + _COT_STEPS
    + "---\n\n"
    "After completing all steps, output ONLY a valid JSON object:\n"
    + _FINAL_JSON
)

print(f"System prompt : {len(T2_SYSTEM)} chars")
print(f"User prompt   : {len(T2_USER)} chars")
print()
print("─" * 65)
print_wrapped(T2_USER[:650], width=65)
print("... [truncated]")

System prompt : 262 chars
User prompt   : 2190 chars

─────────────────────────────────────────────────────────────────
Analyze this cybersecurity incident using the structured
reasoning steps below.
Work through each step carefully before producing the final JSON
output.

**Incident Report:**
On June 14th, multiple employees in the Finance department
reported receiving an email claiming to be from the CFO,
requesting urgent wire transfers to a new vendor account. The
email contained a spoofed domain (cfo-finance-corp.com) and urged
recipients to act within one hour to avoid 'contract penalties'.
One employee clicked the link and entered their corporate
credentials on a fake Office 365 login page before realizing the
site was illegitimate. IT Security was noti
... [truncated]


In [20]:
result_t2 = run_analysis(T2_SYSTEM, T2_USER, label="T2: Guided CoT")
print()
display_result(result_t2)

[T2: Guided CoT] Sending request...
[T2: Guided CoT] Status  : OK
[T2: Guided CoT] Tokens  : 547 in / 137 out

  T2: Guided CoT
  Incident Type  : Phishing
  Severity       : HIGH
  Confidence     : 0.900
  Summary        : Multiple employees in the Finance department received phishing emails that led
them to click on links and enter their credentials on fake Office 365 login pages. This resulted in
immediate disabling of the employee's account.
  Actions (3):
    1. Notify affected users about the phishing attempt and educate them on how to recognize such attacks.
    2. Review and update security policies related to email authentication and user training.
    3. Improve monitoring tools to detect and respond more quickly to similar phishing attempts.
  Tokens         : 547 in / 137 out


---
## 7. Técnica 3 — Meta Prompting

### Racional de desenho

**Meta prompting** usa o próprio LLM para gerar ou melhorar um prompt e depois aplica esse prompt à tarefa real. Essa abordagem em duas fases pode revelar estruturas de prompt que um engenheiro humano talvez não desenhasse diretamente.

**Fase 1 — Geração do prompt:** pedir ao mesmo modelo open source que desenhe o melhor prompt possível para análise de incidentes.

**Fase 2 — Aplicação do prompt:** usar o prompt gerado (mais o texto do incidente) para a análise efetiva.

### Fluxo de iteração

```
[v1: naive] → [meta: generate optimal prompt] → [v2: generated] → [run analysis]
```

### Comportamento esperado
O prompt gerado pode incorporar escolhas estruturais inesperadas, enquadramentos novos e restrições adicionais, potencialmente melhorando a qualidade final.

In [21]:
META_GEN_SYSTEM = (
    "You are an expert prompt engineer specializing in designing AI prompts for high-stakes "
    "security operations systems. You understand how to elicit accurate, calibrated, and "
    "structured outputs from large language models."
)

META_INCIDENT_PLACEHOLDER = "{{INCIDENT_REPORT}}"

_SCHEMA_SPEC = (
    '{ "incident_type": str, "severity": "low|medium|high|critical", '
    '"confidence": float 0-1, "summary": str, "recommended_actions": list[str] }'
)

def build_structured_meta_prompt(extra_guidance=""):
    guidance_block = ""
    if extra_guidance:
        guidance_block = (
            "\n\n## Additional Guidance\n"
            + extra_guidance
        )

    return (
        "## Role\n"
        "You are a senior cybersecurity analyst responsible for SOC triage, incident classification, "
        "and immediate response recommendations.\n\n"
        "## Objective\n"
        "Analyze the incident report and produce a precise, evidence-based assessment.\n\n"
        "## Analysis Workflow\n"
        "1. Identify the initial attack vector and key indicators.\n"
        "2. Choose the single most likely incident type.\n"
        "3. Assess severity using business impact, scope, urgency, and CIA impact.\n"
        "4. Estimate confidence based only on evidence present in the report.\n"
        "5. Write a concise factual summary in 2-3 sentences.\n"
        "6. List prioritized SOC response actions.\n\n"
        "## Decision Rules\n"
        "- Base conclusions only on evidence from the incident report.\n"
        "- Do not invent tools, attackers, or impacts not supported by the report.\n"
        "- Choose exactly one incident_type.\n"
        "- severity must be exactly one of: low, medium, high, critical.\n"
        "- confidence must be a float between 0.0 and 1.0.\n"
        "- If evidence is incomplete, lower confidence instead of speculating.\n"
        "- recommended_actions must be specific, actionable, and ordered by priority."
        + guidance_block
        + "\n\n## Incident Report\n"
        + META_INCIDENT_PLACEHOLDER
        + "\n\n## Output Requirements\n"
        "Respond with ONLY a valid JSON object. Do not include markdown fences, headings, bullet points, "
        "or explanatory text.\n"
        "Use exactly this schema:\n"
        + _FINAL_JSON
    )


def normalize_meta_prompt(raw_prompt):
    cleaned = raw_prompt.strip()
    cleaned = re.sub(r"^```(?:\w+)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    required_sections = [
        "## Role",
        "## Objective",
        "## Analysis Workflow",
        "## Decision Rules",
        "## Incident Report",
        "## Output Requirements",
    ]
    has_required_structure = all(section in cleaned for section in required_sections)
    has_placeholder = META_INCIDENT_PLACEHOLDER in cleaned
    enforces_json_only = "ONLY a valid JSON object" in cleaned or "Return ONLY JSON" in cleaned

    if has_required_structure and has_placeholder and enforces_json_only:
        return cleaned

    compressed_guidance = " ".join(cleaned.split())
    return build_structured_meta_prompt(extra_guidance=compressed_guidance[:500])


META_GEN_USER = (
    "Design a production-ready prompt for a cybersecurity incident analysis assistant.\n\n"
    "Return a complete prompt, not an answer to the incident.\n"
    "The prompt you write must contain exactly these section headers:\n"
    "## Role\n"
    "## Objective\n"
    "## Analysis Workflow\n"
    "## Decision Rules\n"
    "## Incident Report\n"
    "## Output Requirements\n\n"
    "Requirements for the generated prompt:\n"
    "1. It must classify diverse cybersecurity incidents accurately.\n"
    "2. It must calibrate severity as low/medium/high/critical using evidence.\n"
    "3. It must assign a confidence float from 0.0 to 1.0.\n"
    "4. It must produce a concise factual summary.\n"
    "5. It must produce prioritized SOC response actions.\n"
    "6. It must enforce strict JSON output using this schema:\n"
    "   " + _SCHEMA_SPEC + "\n"
    "7. In the Incident Report section, use the literal placeholder "
    + META_INCIDENT_PLACEHOLDER
    + " exactly once.\n"
    "8. Do not include sample answers, example JSON instances, or markdown fences.\n"
    "9. Do not return commentary about prompt engineering. Return only the final prompt text.\n\n"
    "The prompt must be robust across: Ransomware, Phishing, DDoS, Insider Threat, Credential Theft, "
    "Malware, and Supply Chain attacks."
)

print("Phase 1: Generating optimal analysis prompt via meta-prompting...")
print()

RAW_GENERATED_PROMPT, meta_input_tokens, meta_output_tokens = generate_response_text(
    META_GEN_SYSTEM,
    META_GEN_USER,
    max_new_tokens=512,
)
GENERATED_PROMPT = normalize_meta_prompt(RAW_GENERATED_PROMPT)

print("Meta-prompt generated.")
print(f"Tokens: {meta_input_tokens} in / {meta_output_tokens} out")
print(f"Normalized: {'YES' if GENERATED_PROMPT != RAW_GENERATED_PROMPT else 'NO'}")
print()
print("─" * 65)
print("GENERATED PROMPT:")
print("─" * 65)
print_wrapped(GENERATED_PROMPT)

Phase 1: Generating optimal analysis prompt via meta-prompting...

Meta-prompt generated.
Tokens: 319 in / 89 out
Normalized: YES

─────────────────────────────────────────────────────────────────
GENERATED PROMPT:
─────────────────────────────────────────────────────────────────
## Role
You are a senior cybersecurity analyst responsible for SOC triage, incident classification, and
immediate response recommendations.

## Objective
Analyze the incident report and produce a precise, evidence-based assessment.

## Analysis Workflow
1. Identify the initial attack vector and key indicators.
2. Choose the single most likely incident type.
3. Assess severity using business impact, scope, urgency, and CIA impact.
4. Estimate confidence based only on evidence present in the report.
5. Write a concise factual summary in 2-3 sentences.
6. List prioritized SOC response actions.

## Decision Rules
- Base conclusions only on evidence from the incident report.
- Do not invent tools, attackers, or imp

In [22]:
# Comentário em português: a análise final continua usando prompts em inglês.
T3_SYSTEM = (
    "You are a senior cybersecurity analyst. Follow the user's instructions exactly and return "
    "only a valid JSON object. Never use markdown fences, headings, or explanatory prose."
)

T3_USER = GENERATED_PROMPT.replace(META_INCIDENT_PLACEHOLDER, INCIDENT_TEXT)
if META_INCIDENT_PLACEHOLDER in T3_USER:
    raise ValueError("Meta prompt still contains the incident placeholder after injection.")

print("Phase 2: Running analysis with the generated prompt...")
print()
result_t3 = run_analysis(T3_SYSTEM, T3_USER, label="T3: Meta Prompting")
print()
display_result(result_t3)

Phase 2: Running analysis with the generated prompt...

[T3: Meta Prompting] Sending request...
[T3: Meta Prompting] Status  : OK
[T3: Meta Prompting] Tokens  : 538 in / 122 out

  T3: Meta Prompting
  Incident Type  : Phishing
  Severity       : HIGH
  Confidence     : 0.900
  Summary        : Multiple employees received phishing emails demanding urgent wire transfers to a
fake Office 365 login page. One employee fell victim and had their account disabled.
  Actions (3):
    1. Notify all affected users about the phishing attempt and educate them on how to recognize such scams.
    2. Review and update internal security policies regarding phishing threats.
    3. Implement additional monitoring and alerts for suspicious activity related to financial transactions.
  Tokens         : 538 in / 122 out


---
## 8. Demonstração de Iteração de Prompts

As três técnicas representam uma evolução desde um prompt ingênuo até desenhos progressivamente mais sofisticados.

```
v1 (Base) ──► v2 (RCTOF) ──► v3 (Guided CoT) ──► v4 (Meta-gerado)
    ↓              ↓                 ↓                    ↓
Sem estrutura  Papel explícito   Raciocínio guiado   Estrutura desenhada pelo LLM
Sem schema     + schema JSON     passo a passo       automaticamente
```

In [23]:
V1_PROMPT = (
    "Analyze this cybersecurity incident and output a JSON object with: "
    "incident_type, severity, confidence, summary, recommended_actions.\n\n"
    + INCIDENT_TEXT
)

prompts = [
    ("v1 — Baseline (no structure)", V1_PROMPT),
    ("v2 — RCTOF (Technique 1)", T1_USER),
    ("v3 — Guided CoT (Technique 2)", T2_USER),
    ("v4 — Meta-generated (Technique 3)", T3_USER),
]

for label, prompt in prompts:
    print(f"{'─' * 65}")
    print(f"  {label}")
    print(f"  Length: {len(prompt)} chars")
    print()
    preview = " ".join(prompt[:280].split())
    print_wrapped(f"  Preview: {preview}...")
    print()

─────────────────────────────────────────────────────────────────
  v1 — Baseline (no structure)
  Length: 689 chars

  Preview: Analyze this cybersecurity incident and output a JSON object with: incident_type,
severity, confidence, summary, recommended_actions. On June 14th, multiple employees in the Finance
department reported receiving an email claiming to be from the CFO, requesting urgent wire transf...

─────────────────────────────────────────────────────────────────
  v2 — RCTOF (Technique 1)
  Length: 1327 chars

  Preview: ## Context Your organization's SOC has received a cybersecurity incident report requiring
immediate triage. ## Task Analyze the incident report below. Identify the attack type, assess
severity, estimate your confidence level, write a concise summary, and list prioritized respons...

─────────────────────────────────────────────────────────────────
  v3 — Guided CoT (Technique 2)
  Length: 2190 chars

  Preview: Analyze this cybersecurity incident using the 

---
## 9. Tabela Comparativa

Resultados lado a lado das três técnicas sobre o mesmo incidente.

In [24]:
all_results = [result_t1, result_t2, result_t3]

rows = []
for r in all_results:
    analysis = r.get("analysis")
    rows.append({
        "Technique": r["label"],
        "Parse OK": "YES" if r["parse_ok"] else "NO",
        "Valid": "YES" if r["validation_ok"] else "NO",
        "Incident Type": analysis.incident_type if analysis else "—",
        "Severity": analysis.severity.value if analysis else "—",
        "Confidence": f"{analysis.confidence:.3f}" if analysis else "—",
        "# Actions": len(analysis.recommended_actions) if analysis else 0,
        "In Tok": r["input_tokens"],
        "Out Tok": r["output_tokens"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()

# Compara a consistência geral apenas quando todas as saídas passam na validação.
if all(r["validation_ok"] for r in all_results):
    types = {r["analysis"].incident_type for r in all_results}
    severities = {r["analysis"].severity.value for r in all_results}
    confidences = [r["analysis"].confidence for r in all_results]

    print(f"Incident type agreement : {types}")
    print(f"Severity agreement      : {severities}")
    print(f"Confidence range        : {min(confidences):.3f} – {max(confidences):.3f}")
    print(f"Confidence spread       : {max(confidences) - min(confidences):.3f}")

         Technique Parse OK Valid Incident Type Severity Confidence  # Actions  In Tok  Out Tok
         T1: RCTOF      YES   YES      Phishing     high      0.900          2     327       94
    T2: Guided CoT      YES   YES      Phishing     high      0.900          3     547      137
T3: Meta Prompting      YES   YES      Phishing     high      0.900          3     538      122

Incident type agreement : {'Phishing'}
Severity agreement      : {'high'}
Confidence range        : 0.900 – 0.900
Confidence spread       : 0.000


---
## 10. Discussão

### Técnica 1 — RCTOF

**Pontos fortes:**
- Overhead mínimo de tokens, com uma única geração e prompt curto
- Estrutura previsível de saída, pois o esquema é imposto no próprio prompt
- Boa aderência a pipelines de classificação em alto volume e sensíveis a latência

**Limitações:**
- Não oferece trilha explícita de raciocínio para auditoria
- Pode falhar silenciosamente em incidentes ambíguos, com confiança inflada
- Aderência ao schema depende do wording do prompt e do comportamento do modelo

---

### Técnica 2 — Guided Chain of Thought

**Pontos fortes:**
- Tendência a melhor calibração de confiança, já que o prompt força avaliação baseada em evidências
- Melhores notas de severidade, pois a tríade CIA fornece uma base mais disciplinada
- Lida melhor com incidentes ambíguos, fazendo a incerteza aparecer com mais naturalidade

**Limitações:**
- Custo maior em tokens por causa do prompt mais longo
- Latência maior em comparação com a técnica RCTOF
- Pode ser excessiva para incidentes muito claros e diretos

---

### Técnica 3 — Meta Prompting

**Pontos fortes:**
- Revela estruturas de prompt que o engenheiro talvez não tivesse considerado
- Útil como ferramenta de descoberta para novas categorias de incidente ou novos domínios
- Permite reexecutar a etapa meta quando os tipos de incidente ou o contexto organizacional mudarem

**Limitações:**
- A geração do prompt é não determinística e pode variar entre execuções
- Exige duas gerações, aumentando custo computacional e tempo total
- O prompt gerado ainda precisa de revisão humana antes de ir para produção
- Pode introduzir restrições alucinadas ou enquadramentos subótimos

---

### Validação de Schema com Pydantic

O uso de Pydantic fornece:
- **Mensagens de erro por campo** para depurar respostas malformadas
- **Enforcement de Enum** em `severity`, bloqueando strings inválidas antes de propagarem
- **Validação de intervalo** em `confidence`, evitando floats fora da faixa esperada
- Um contrato de schema reutilizável e versionável, independente da técnica de prompting

A cadeia de fallback em `parse_json_response` (direto → remoção de fences → regex gulosa) cobre os modos de falha mais comuns sem sacrificar precisão.

---
## 11. Conclusão

Este notebook demonstrou três técnicas de prompt engineering aplicadas à triagem de incidentes de cibersegurança com um **modelo open source do Hugging Face**:

| Técnica | Melhor encaixe | Compromisso principal |
|---------|----------------|-----------------------|
| **RCTOF** | Pipelines de alto volume, incidentes claros | Rápida e simples, mas sem trilha de raciocínio |
| **Guided CoT** | Incidentes ambíguos, decisões sensíveis à calibração | Mais consistente, porém com prompts mais longos |
| **Meta Prompting** | Descoberta de prompts, novas categorias | Estrutura nova, porém com variabilidade entre execuções |

### Principais conclusões

1. **A estrutura do prompt importa mais do que o tamanho.** Um schema bem especificado reduz a maior parte das falhas de parsing de JSON, independentemente da técnica.

2. **O raciocínio guiado tende a melhorar a calibração.** As notas de confiança e severidade ficam mais ancoradas nas evidências quando o prompt força uma passagem disciplinada pela tríade CIA.

3. **A validação com Pydantic é uma barreira rígida.** Sistemas SOC a não devem processar saídas de LLM sem validação prévia; o schema é o contrato entre o modelo e o pipeline.

4. **Meta prompting é ferramenta de pesquisa, não atalho de produção.** Prompts gerados precisam de revisão humana e versionamento como qualquer outro artefato de engenharia.

# Cybersecurity Incident Analyzer — Fase 3

## Semantic Embeddings e Information Retrieval

Esta fase estende o projeto com uma camada de **recuperação semântica** (semantic retrieval), capaz de encontrar os trechos de documentos mais relevantes de uma base de conhecimento de cibersegurança para uma consulta em linguagem natural.

O pipeline implementado segue o fluxo abaixo:

```text
PDF URL -> Download -> Unstructured (LangChain) -> Markdown -> LangChain Loader -> Chunking -> Embeddings -> ChromaDB
```

O objetivo aqui é construir um pipeline de recuperação semântica robusto, que sirva de base para o question answering fundamentado em documentos nas próximas fases.


---
## 1. Instalação de Dependências e Imports

Esta fase introduz um conjunto novo de bibliotecas, todas open source:

- **Unstructured** (via LangChain): responsável por toda a ingestão de documentos, convertendo PDF em Markdown estruturado;
- **LangChain**: orquestra o carregamento de Markdown, o chunking, a geração de embeddings e a indexação vetorial;
- **Sentence Transformers** (via Hugging Face): geração de embeddings semânticos;
- **ChromaDB**: banco de dados vetorial usado para indexação e busca por similaridade.

As bibliotecas das fases anteriores (`pandas`, `numpy`) são reaproveitadas.


In [25]:
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display

from langchain_community.document_loaders import UnstructuredMarkdownLoader, UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)

print("Bibliotecas da Fase 3 carregadas com sucesso.")


/tmp/ipykernel_553/1009299217.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredMarkdownLoader, UnstructuredPDFLoader


Bibliotecas da Fase 3 carregadas com sucesso.


---
## 2. Estrutura de Pastas e Configuração dos Documentos

A base de conhecimento é organizada em duas pastas dentro de `knowledge_base/`:

- `knowledge_base/pdf/`: PDFs originais baixados;
- `knowledge_base/markdown/`: versões Markdown geradas a partir dos PDFs.

A lista `DOCUMENTS` define a base de conhecimento, cobrindo os principais frameworks e taxonomias de referência da indústria (NIST, OWASP, MITRE ATT&CK, CISA), conforme discutido em `docs/recommended_corpus.md`.

> A edição 2021 do OWASP Top 10 não possui mais um PDF oficial único — o conteúdo passou a ser publicado apenas em formato web multi-página. Por isso, usamos a última edição do OWASP Top 10 que ainda existe como PDF oficial único (2017).


In [26]:
KNOWLEDGE_BASE_DIR = Path("knowledge_base")
PDF_DIR = KNOWLEDGE_BASE_DIR / "pdf"
MARKDOWN_DIR = KNOWLEDGE_BASE_DIR / "markdown"
CHROMA_DIR = KNOWLEDGE_BASE_DIR / "chroma_db"

for directory in (PDF_DIR, MARKDOWN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Diretorio de PDFs: {PDF_DIR.resolve()}")
print(f"Diretorio de Markdown: {MARKDOWN_DIR.resolve()}")

DOCUMENTS = [
    {
        "name": "NIST Cybersecurity Framework (CSF) 2.0",
        "category": "Risk Management",
        "url": "https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
    },
    {
        "name": "OWASP Top 10 (2017)",
        "category": "Application Security",
        "url": "https://wiki.owasp.org/images/7/72/OWASP_Top_10-2017_%28en%29.pdf.pdf",
    },
    {
        "name": "MITRE ATT&CK - Design and Philosophy",
        "category": "Threat Taxonomy",
        "url": "https://attack.mitre.org/docs/ATTACK_Design_and_Philosophy_March_2020.pdf",
    },
    {
        "name": "CISA Cybersecurity Incident and Vulnerability Response Playbooks",
        "category": "Incident Response",
        "url": "https://www.cisa.gov/sites/default/files/publications/Federal_Government_Cybersecurity_Incident_and_Vulnerability_Response_Playbooks_508C.pdf",
    },
]

pd.DataFrame(DOCUMENTS)[["name", "category", "url"]]

Diretorio de PDFs: /content/knowledge_base/pdf
Diretorio de Markdown: /content/knowledge_base/markdown


,name,category,url
0,NIST Cybersecurity Framework (CSF) 2.0,Risk Management,https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf
1,OWASP Top 10 (2017),Application Security,https://wiki.owasp.org/images/7/72/OWASP_Top_10-2017_%28en%29.pdf.pdf
2,MITRE ATT&CK - Design and Philosophy,Threat Taxonomy,https://attack.mitre.org/docs/ATTACK_Design_and_Philosophy_March_2020.pdf
3,CISA Cybersecurity Incident and Vulnerability Response Playbooks,Incident Response,https://www.cisa.gov/sites/default/files/publications/Federal_Government_Cybersecurity_Incident_and_Vulnerability_Response_Playbooks_508C.pdf


---
## 3. Pipeline de Ingestao de Documentos

O pipeline de ingestao segue estas etapas:

| Etapa | Responsabilidade |
|---|---|
| **Download** | Baixa cada PDF da lista `DOCUMENTS` para `knowledge_base/pdf/` |
| **Unstructured (LangChain)** | Converte PDF em Markdown preservando estrutura (titulos, listas, tabelas) |
| **LangChain Loader** | Carrega os arquivos Markdown como objetos `Document`, com metadados |
| **Chunking** | Divide cada documento em trechos menores e semanticamente coerentes |
| **Embeddings** | Converte cada chunk em um vetor denso usando Sentence Transformers |
| **ChromaDB** | Indexa os vetores para busca por similaridade |

Cada etapa e implementada como uma funcao reutilizavel, evitando duplicacao de codigo.


---
## 4. Etapa 1 — Download dos PDFs

A funcao `download_documents()` baixa cada PDF listado em `DOCUMENTS`, salva em `knowledge_base/pdf/` e trata falhas de rede sem interromper o pipeline. Downloads ja existentes em disco sao reaproveitados, permitindo reexecutar o notebook sem baixar tudo novamente.

In [27]:
def _slugify(name):
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_").lower()
    return slug[:80]


def download_documents(documents, pdf_dir, timeout=60):
    results = []
    for doc in documents:
        filename = f"{_slugify(doc['name'])}.pdf"
        pdf_path = pdf_dir / filename
        status = "skipped (already downloaded)"

        if not pdf_path.exists():
            try:
                response = requests.get(
                    doc["url"], timeout=timeout, headers={"User-Agent": "Mozilla/5.0"}
                )
                response.raise_for_status()
                pdf_path.write_bytes(response.content)
                status = "downloaded"
            except requests.RequestException as exc:
                status = f"failed ({exc})"
                pdf_path = None

        results.append(
            {
                "name": doc["name"],
                "category": doc["category"],
                "url": doc["url"],
                "pdf_path": str(pdf_path) if pdf_path else None,
                "status": status,
            }
        )

    return results


download_results = download_documents(DOCUMENTS, PDF_DIR)
pd.DataFrame(download_results)[["name", "category", "status"]]

,name,category,status
0,NIST Cybersecurity Framework (CSF) 2.0,Risk Management,downloaded
1,OWASP Top 10 (2017),Application Security,downloaded
2,MITRE ATT&CK - Design and Philosophy,Threat Taxonomy,downloaded
3,CISA Cybersecurity Incident and Vulnerability Response Playbooks,Incident Response,downloaded


---
## 5. Etapa 2 — Conversao de PDF para Markdown com Unstructured (LangChain)

A biblioteca **Unstructured**, atraves do `UnstructuredPDFLoader` do LangChain, e responsavel por toda a etapa de ingestao de documentos, convertendo o layout do PDF em elementos estruturados (titulos, paragrafos, listas, tabelas), que reconstruimos como Markdown.

### Por que usar Markdown como representacao intermediaria?

- **Preserva hierarquia**: elementos identificados como titulo viram cabecalhos `##`, mantendo a estrutura logica do documento;
- **Preserva listas**: itens de lista continuam identificaveis como listas;
- **Melhora o chunking semantico**: com a estrutura (titulos, secoes) explicita no texto, o splitter corta o documento em pontos mais coerentes semanticamente, em vez de cortar no meio de uma frase sem contexto.

Sem essa etapa, o texto extraido diretamente de um PDF frequentemente perde a ordem de leitura e mistura cabecalhos, rodapes e colunas de forma desorganizada.

> A extracao de layout roda modelos localmente (via `unstructured`) e pode levar alguns minutos por documento em CPU.


In [28]:
import traceback

from unstructured.partition.utils.config import env_config

if not hasattr(env_config, "PDF_RENDER_MAX_PIXELS_PER_PAGE"):
    # Some unstructured builds reference this cap in the hi_res/OCR image-rendering
    # path without it being defined on ENVConfig; set a generous value so it never
    # triggers a downscale, instead of crashing with AttributeError.
    env_config.PDF_RENDER_MAX_PIXELS_PER_PAGE = 300_000_000


def _extract_elements(pdf_path, strategy):
    return UnstructuredPDFLoader(str(pdf_path), mode="elements", strategy=strategy).load()


def convert_pdf_to_markdown(pdf_path, markdown_dir, min_chars=200):
    pdf_path = Path(pdf_path)
    markdown_path = markdown_dir / f"{pdf_path.stem}.md"

    if markdown_path.exists():
        return markdown_path

    fast_start = time.time()
    elements = _extract_elements(pdf_path, strategy="fast")
    extracted_chars = sum(len(el.page_content.strip()) for el in elements)
    print(f"  [fast] {extracted_chars} chars em {time.time() - fast_start:.1f}s", flush=True)

    if extracted_chars < min_chars:
        # "fast" (pdfminer text-layer extraction) came back empty or too short,
        # likely due to embedded/subset fonts without a proper text layer.
        # Fall back to "hi_res" (layout model + OCR), which does not depend on
        # the PDF's text layer at all.
        print(f"  [fast] abaixo do minimo ({min_chars}), tentando hi_res (mais lento)...", flush=True)
        hires_start = time.time()
        elements = _extract_elements(pdf_path, strategy="hi_res")
        print(f"  [hi_res] concluido em {time.time() - hires_start:.1f}s", flush=True)

    lines = []
    for element in elements:
        text = element.page_content.strip()
        if not text:
            continue

        category = element.metadata.get("category")
        if category == "Title":
            lines.append(f"## {text}")
        elif category == "ListItem":
            lines.append(f"- {text}")
        else:
            lines.append(text)

    markdown_text = "\n\n".join(lines)
    markdown_path.write_text(markdown_text, encoding="utf-8")

    return markdown_path


conversion_results = []
for i, doc in enumerate(download_results, start=1):
    print(f"[{i}/{len(download_results)}] {doc['name']}", flush=True)
    if doc["pdf_path"] is None:
        conversion_results.append({**doc, "markdown_path": None, "conversion_status": "skipped (no PDF)"})
        continue

    try:
        start = time.time()
        markdown_path = convert_pdf_to_markdown(doc["pdf_path"], MARKDOWN_DIR)
        elapsed = time.time() - start
        conversion_results.append(
            {**doc, "markdown_path": str(markdown_path), "conversion_status": f"converted ({elapsed:.1f}s)"}
        )
    except Exception as exc:
        print(f"=== Falha ao converter: {doc['name']} ===")
        traceback.print_exc()
        conversion_results.append({**doc, "markdown_path": None, "conversion_status": f"failed ({exc})"})

pd.DataFrame(conversion_results)[["name", "category", "conversion_status"]]


[1/4] NIST Cybersecurity Framework (CSF) 2.0


  [fast] 67644 chars em 12.6s
[2/4] OWASP Top 10 (2017)


  [fast] 92303 chars em 9.0s
[3/4] MITRE ATT&CK - Design and Philosophy


  [fast] 99333 chars em 5.9s
[4/4] CISA Cybersecurity Incident and Vulnerability Response Playbooks


  [fast] 101651 chars em 10.7s


,name,category,conversion_status
0,NIST Cybersecurity Framework (CSF) 2.0,Risk Management,converted (12.6s)
1,OWASP Top 10 (2017),Application Security,converted (9.0s)
2,MITRE ATT&CK - Design and Philosophy,Threat Taxonomy,converted (5.9s)
3,CISA Cybersecurity Incident and Vulnerability Response Playbooks,Incident Response,converted (10.7s)


In [29]:
example_markdown_path = next((doc["markdown_path"] for doc in conversion_results if doc["markdown_path"]), None)

if example_markdown_path:
    example_markdown_text = Path(example_markdown_path).read_text(encoding="utf-8")
    print(f"Exemplo de Markdown gerado a partir de: {example_markdown_path}\n")
    print_wrapped(example_markdown_text[:1500])

Exemplo de Markdown gerado a partir de: knowledge_base/markdown/nist_cybersecurity_framework_csf_2_0.md

## The NIST Cybersecurity Framework (CSF) 2.0

National Institute of Standards and Technology This publication is available free of charge from:
https://doi.org/10.6028/NIST.CSWP.29

## February 26, 2024

NIST CSWP 29 February 26, 2024

The NIST Cybersecurity Framework (CSF) 2.0

## Abstract

The NIST Cybersecurity Framework (CSF) 2.0 provides guidance to industry, government agencies, and
other organizations to manage cybersecurity risks. It offers a taxonomy of high- level cybersecurity
outcomes that can be used by any organization — regardless of its size, sector, or maturity — to
better understand, assess, prioritize, and communicate its cybersecurity efforts. The CSF does not
prescribe how outcomes should be achieved. Rather, it links to online resources that provide
additional guidance on practices and controls that could be used to achieve those outcomes. This
document descri

---
## 6. Etapa 3 — Carregamento dos Documentos com LangChain

Com os arquivos Markdown gerados, usamos o `UnstructuredMarkdownLoader` do LangChain para carregar cada arquivo como um objeto `Document`, anexando metadados (nome do documento, categoria e caminho de origem) que serao propagados ate os chunks e, posteriormente, ate os resultados de busca.

In [30]:
def load_markdown_documents(conversion_results):
    documents = []
    for doc in conversion_results:
        if not doc.get("markdown_path"):
            continue

        loader = UnstructuredMarkdownLoader(doc["markdown_path"], mode="single")
        loaded = loader.load()

        for loaded_doc in loaded:
            loaded_doc.metadata.update(
                {
                    "document_name": doc["name"],
                    "category": doc["category"],
                    "markdown_source": doc["markdown_path"],
                }
            )
            documents.append(loaded_doc)

    return documents


loaded_documents = load_markdown_documents(conversion_results)

print(f"Documentos carregados: {len(loaded_documents)}")
pd.DataFrame([doc.metadata for doc in loaded_documents])[["document_name", "category", "markdown_source"]]

Documentos carregados: 4


,document_name,category,markdown_source
0,NIST Cybersecurity Framework (CSF) 2.0,Risk Management,knowledge_base/markdown/nist_cybersecurity_framework_csf_2_0.md
1,OWASP Top 10 (2017),Application Security,knowledge_base/markdown/owasp_top_10_2017.md
2,MITRE ATT&CK - Design and Philosophy,Threat Taxonomy,knowledge_base/markdown/mitre_att_ck_design_and_philosophy.md
3,CISA Cybersecurity Incident and Vulnerability Response Playbooks,Incident Response,knowledge_base/markdown/cisa_cybersecurity_incident_and_vulnerability_response_playbooks.md


---
## 7. Etapa 4 — Chunking

Modelos de embedding produzem representacoes mais precisas quando o texto de entrada e curto e semanticamente coeso. Por isso, cada documento e dividido em **chunks** menores antes de gerar embeddings, usando `RecursiveCharacterTextSplitter`.

### Por que o chunking e necessario?

- documentos completos (dezenas de paginas) nao cabem em um unico vetor de embedding sem perder granularidade;
- consultas especificas devem recuperar trechos focados, nao o documento inteiro;
- chunks menores melhoram a precisao da busca por similaridade.

### Por que usar overlap?

O `chunk_overlap` garante que uma frase ou ideia dividida entre dois chunks nao perca contexto — o final de um chunk se repete no inicio do proximo, evitando que informacao relevante seja cortada exatamente na fronteira entre dois trechos.

### Trade-offs entre chunks maiores e menores

| | Chunks menores | Chunks maiores |
|---|---|---|
| **Precisao da busca** | Mais precisa, foco em um unico conceito | Pode misturar multiplos topicos |
| **Contexto preservado** | Pode perder contexto ao redor | Preserva mais contexto |
| **Custo computacional** | Mais vetores para indexar e buscar | Menos vetores, busca mais rapida |
| **Risco de ruido** | Baixo | Maior risco de incluir trechos irrelevantes |

Usamos `chunk_size=1000` e `chunk_overlap=150`, um equilibrio comum para documentos tecnicos em linguagem natural.

In [31]:
def split_documents(documents, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)

    for position, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = position
        heading_match = re.findall(r"^#{1,6}\s+.*$", chunk.page_content, flags=re.MULTILINE)
        chunk.metadata["section_heading"] = heading_match[0] if heading_match else "N/A"
        chunk.metadata["page_number"] = "N/A"

    return chunks


chunks = split_documents(loaded_documents, chunk_size=1000, chunk_overlap=150)

print(f"Total de chunks gerados: {len(chunks)}")
for chunk in chunks[:2]:
    print("-" * 100)
    print(f"Documento: {chunk.metadata['document_name']} | Chunk ID: {chunk.metadata['chunk_id']}")
    print_wrapped(chunk.page_content)

Total de chunks gerados: 484
----------------------------------------------------------------------------------------------------
Documento: NIST Cybersecurity Framework (CSF) 2.0 | Chunk ID: 0
The NIST Cybersecurity Framework (CSF) 2.0

National Institute of Standards and Technology This publication is available free of charge from:
https://doi.org/10.6028/NIST.CSWP.29

February 26, 2024

NIST CSWP 29 February 26, 2024

The NIST Cybersecurity Framework (CSF) 2.0

Abstract

The NIST Cybersecurity Framework (CSF) 2.0 provides guidance to industry, government agencies, and
other organizations to manage cybersecurity risks. It offers a taxonomy of high- level cybersecurity
outcomes that can be used by any organization — regardless of its size, sector, or maturity — to
better understand, assess, prioritize, and communicate its cybersecurity efforts. The CSF does not
prescribe how outcomes should be achieved. Rather, it links to online resources that provide
additional guidance on practices

---
## 8. Etapa 5 — Embeddings Semanticos

### O que sao sentence embeddings?

Um modelo de sentence embeddings converte um texto (uma frase, um paragrafo ou um chunk) em um vetor denso de números reais, onde a posição no espaço vetorial captura o **significado** do texto, nao apenas as palavras usadas literalmente.

### Similaridade semantica vs. busca por palavra-chave

- **Busca por palavra-chave** (ex: TF-IDF, BM25) depende de correspondencia exata (ou aproximada) de termos. Uma busca por "phishing" nao encontraria facilmente um trecho que fala apenas em "fraudulent email impersonating an executive", mesmo tratando do mesmo conteudo.
- **Busca semantica** compara vetores no espaco de embeddings. Textos com significados parecidos ficam proximos nesse espaco, mesmo usando vocabulario diferente.

### Por que `sentence-transformers/all-MiniLM-L6-v2`?

- e um modelo **open source** do Hugging Face, alinhado a politica do projeto;
- e leve (6 camadas) e roda bem em CPU, sem exigir GPU;
- produz embeddings de qualidade competitiva para tarefas de retrieval em ingles, o idioma dos documentos da base de conhecimento;
- e um dos baselines mais usados em benchmarks de sentence embeddings (ex: MTEB) para retrieval.

In [32]:
def generate_embeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"):
    return HuggingFaceEmbeddings(model_name=model_name)


embedding_model = generate_embeddings()

chunk_texts = [chunk.page_content for chunk in chunks]
chunk_vectors = np.array(embedding_model.embed_documents(chunk_texts))

print("Modelo de embeddings: sentence-transformers/all-MiniLM-L6-v2")
print(f"Dimensao do embedding: {chunk_vectors.shape[1]}")
print(f"Total de vetores criados: {chunk_vectors.shape[0]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings: sentence-transformers/all-MiniLM-L6-v2
Dimensao do embedding: 384
Total de vetores criados: 484


---
## 9. Etapa 6 — Banco de Dados Vetorial (ChromaDB)

O **ChromaDB** indexa os vetores de embedding junto com o texto original e os metadados de cada chunk, permitindo busca por similaridade eficiente sem comparar a consulta contra todos os vetores manualmente (o que fazemos de proposito, de forma manual, na Etapa 8, para fins de comparacao).

### Por que metadados importam para recuperacao?

Metadados como nome do documento, categoria, arquivo de origem, `chunk_id` e cabecalho de secao permitem:

- **rastrear a origem** de cada resultado retornado, essencial para auditoria e confianca em um sistema de seguranca;
- **filtrar buscas** por categoria (ex: restringir a busca a documentos de "Incident Response") em fases futuras;
- **explicar o resultado** ao analista humano, mostrando nao so o texto recuperado, mas de qual secao de qual documento ele veio.

In [33]:
def create_vector_store(chunks, embedding_model, persist_directory, collection_name="cybersecurity_kb"):
    return Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name=collection_name,
        persist_directory=str(persist_directory),
        collection_metadata={"hnsw:space": "cosine"},
    )


vector_store = create_vector_store(chunks, embedding_model, CHROMA_DIR)

print(f"Colecao Chroma criada com {vector_store._collection.count()} vetores.")

Colecao Chroma criada com 484 vetores.


---
## 10. Etapa 7 — Recuperacao Semantica via ChromaDB

A funcao `semantic_search()` implementa o fluxo completo de recuperacao:

```text
Consulta em linguagem natural -> Embedding da consulta -> Busca por similaridade no Chroma -> Top-5 chunks mais relevantes
```

O espaco de distancia da colecao foi configurado como cosseno (`hnsw:space=cosine`), entao o score retornado ja e uma similaridade (quanto maior, mais relevante).

In [34]:
def semantic_search(vector_store, query, k=5):
    results = vector_store.similarity_search_with_relevance_scores(query, k=k)
    return [
        {
            "query": query,
            "document_name": doc.metadata.get("document_name"),
            "category": doc.metadata.get("category"),
            "chunk": doc.page_content,
            "similarity_score": score,
            "section_heading": doc.metadata.get("section_heading"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "strategy": "chroma",
        }
        for doc, score in results
    ]


example_query = "How should I respond to a phishing attack?"
example_chroma_results = semantic_search(vector_store, example_query, k=5)

for rank, result in enumerate(example_chroma_results, start=1):
    print("-" * 100)
    print(f"[{rank}] {result['document_name']} | {result['category']} | score={result['similarity_score']:.4f}")
    print_wrapped(result["chunk"])


----------------------------------------------------------------------------------------------------
[1] NIST Cybersecurity Framework (CSF) 2.0 | Risk Management | score=0.3776
DETECT (DE) — Possible cybersecurity attacks and compromises are found and analyzed. DETECT enables
the timely discovery and analysis of anomalies, indicators of compromise, and other potentially
adverse events that may indicate that cybersecurity attacks and incidents are occurring. This
Function supports successful incident response and recovery activities.

RESPOND (RS) — Actions regarding a detected cybersecurity incident are taken. RESPOND supports the
ability to contain the effects of cybersecurity incidents. Outcomes within this Function cover
incident management, analysis, mitigation, reporting, and communication.

RECOVER (RC) — Assets and operations affected by a cybersecurity incident are restored. RECOVER
supports the timely restoration of normal operations to reduce the effects of cybersecurity
inci

---
## 11. Etapa 8 — Similaridade de Cosseno Manual

Como segunda estrategia de recuperacao, implementamos a busca por similaridade de cosseno **manualmente**, com NumPy, reaproveitando exatamente os mesmos vetores de embedding (`chunk_vectors`) gerados na Etapa 5 — sem gerar novos embeddings.

### Similaridade de cosseno

A similaridade de cosseno mede o angulo entre dois vetores, ignorando sua magnitude:

```text
cosine_similarity(A, B) = (A . B) / (||A|| * ||B||)
```

Quanto mais proximo de 1, mais semanticamente similares os textos.

In [35]:
def cosine_similarity_search(query, embedding_model, chunk_vectors, chunks, k=5):
    query_vector = np.array(embedding_model.embed_query(query))

    chunk_norms = np.linalg.norm(chunk_vectors, axis=1)
    query_norm = np.linalg.norm(query_vector)
    similarities = (chunk_vectors @ query_vector) / (chunk_norms * query_norm + 1e-10)

    top_indices = np.argsort(similarities)[::-1][:k]

    return [
        {
            "query": query,
            "document_name": chunks[idx].metadata.get("document_name"),
            "category": chunks[idx].metadata.get("category"),
            "chunk": chunks[idx].page_content,
            "similarity_score": float(similarities[idx]),
            "section_heading": chunks[idx].metadata.get("section_heading"),
            "chunk_id": chunks[idx].metadata.get("chunk_id"),
            "strategy": "manual_cosine",
        }
        for idx in top_indices
    ]


example_manual_results = cosine_similarity_search(example_query, embedding_model, chunk_vectors, chunks, k=5)

for rank, result in enumerate(example_manual_results, start=1):
    print("-" * 100)
    print(f"[{rank}] {result['document_name']} | {result['category']} | score={result['similarity_score']:.4f}")
    print_wrapped(result["chunk"])


----------------------------------------------------------------------------------------------------
[1] NIST Cybersecurity Framework (CSF) 2.0 | Risk Management | score=0.3776
DETECT (DE) — Possible cybersecurity attacks and compromises are found and analyzed. DETECT enables
the timely discovery and analysis of anomalies, indicators of compromise, and other potentially
adverse events that may indicate that cybersecurity attacks and incidents are occurring. This
Function supports successful incident response and recovery activities.

RESPOND (RS) — Actions regarding a detected cybersecurity incident are taken. RESPOND supports the
ability to contain the effects of cybersecurity incidents. Outcomes within this Function cover
incident management, analysis, mitigation, reporting, and communication.

RECOVER (RC) — Assets and operations affected by a cybersecurity incident are restored. RECOVER
supports the timely restoration of normal operations to reduce the effects of cybersecurity
inci

---
## 12. Consultas de Teste

Definimos dez consultas realistas de ciberseguranca, alinhadas aos tipos de incidentes ja usados nas Fases 1 e 2 (`incidents`), para avaliar as duas estrategias de recuperacao.

In [36]:
TEST_QUERIES = [
    "How should I respond to a phishing attack?",
    "How can ransomware be contained?",
    "What are the phases of incident response?",
    "How should SQL Injection be mitigated?",
    "How can brute-force attacks be detected?",
    "What are indicators of malware infection?",
    "How should privileged accounts be protected?",
    "How is credential theft mitigated?",
    "How should data exfiltration incidents be handled?",
    "What is the recommended response to suspicious login attempts?",
]

chroma_results = []
manual_results = []

for query in TEST_QUERIES:
    chroma_results.extend(semantic_search(vector_store, query, k=5))
    manual_results.extend(cosine_similarity_search(query, embedding_model, chunk_vectors, chunks, k=5))

print(f"Consultas executadas: {len(TEST_QUERIES)}")
print(f"Resultados Chroma: {len(chroma_results)} | Resultados cosine manual: {len(manual_results)}")

Consultas executadas: 10
Resultados Chroma: 50 | Resultados cosine manual: 50


---
## 13. Avaliacao de Recuperacao

As tabelas abaixo mostram, para cada consulta, o Top-5 de chunks recuperados por cada estrategia, com documento de origem, categoria e score de similaridade.

In [37]:
def build_results_table(results):
    df = pd.DataFrame(results)
    df["rank"] = df.groupby(["query", "strategy"]).cumcount() + 1
    return df[["query", "strategy", "rank", "document_name", "category", "similarity_score", "chunk"]]


def print_query_results(df, query):
    subset = df[df["query"] == query].sort_values(["strategy", "rank"])
    for _, row in subset.iterrows():
        print("-" * 100)
        print(
            f"[{row['strategy']}] rank {row['rank']} | {row['document_name']} | "
            f"{row['category']} | score={row['similarity_score']:.4f}"
        )
        print_wrapped(row["chunk"])


df_chroma = build_results_table(chroma_results)
df_manual = build_results_table(manual_results)
df_comparison = pd.concat([df_chroma, df_manual], ignore_index=True)

for query in TEST_QUERIES[:3]:
    print("=" * 100)
    print(f"Query: {query}")
    print_query_results(df_comparison, query)


Query: How should I respond to a phishing attack?
----------------------------------------------------------------------------------------------------
[chroma] rank 1 | NIST Cybersecurity Framework (CSF) 2.0 | Risk Management | score=0.3776
DETECT (DE) — Possible cybersecurity attacks and compromises are found and analyzed. DETECT enables
the timely discovery and analysis of anomalies, indicators of compromise, and other potentially
adverse events that may indicate that cybersecurity attacks and incidents are occurring. This
Function supports successful incident response and recovery activities.

RESPOND (RS) — Actions regarding a detected cybersecurity incident are taken. RESPOND supports the
ability to contain the effects of cybersecurity incidents. Outcomes within this Function cover
incident management, analysis, mitigation, reporting, and communication.

RECOVER (RC) — Assets and operations affected by a cybersecurity incident are restored. RECOVER
supports the timely restoration 

In [38]:
df_comparison.sort_values(["query", "strategy", "rank"]).reset_index(drop=True)

,query,strategy,rank,document_name,category,similarity_score,chunk
0,How can brute-force attacks be detected?,chroma,1,MITRE ATT&CK - Design and Philosophy,Threat Taxonomy,0.400625,"©2020 The MITRE Corporation. All Rights Reserved\n\nApproved for Public Release. Distribution unlimited 19-01075-28.\n\n3.4 Techniques and Sub-Techniques Techniques represent “how” an adversary achieves a tactical objective by performing an action. For example, an adversary may dump credentials from an operating system to gain access to useful credentials within a network. Techniques may also represent “what” an adversary gains by performing an action. This is a useful distinction for the Discovery tactic as the techniques highlight what type of information an adversary is after with a particular action."
1,How can brute-force attacks be detected?,chroma,2,OWASP Top 10 (2017),Application Security,0.391814,"Attack Vectors\n\nSecurity Weakness\n\nImpacts\n\nApp. Specific\n\nExploitability: 2\n\nPrevalence: 3\n\nDetectability: 2\n\nTechnical: 3\n\nBusiness ?\n\nRather than directly attacking crypto, attackers steal keys, execute man-in- the-middle attacks, or steal clear text data off the server, while in transit, or from the user’s client, e.g. browser. A manual attack is generally required. Previously retrieved password databases could be brute forced by Graphics Processing Units (GPUs).\n\nOver the last few years, this has been the most common impactful attack. The most common flaw is simply not encrypting sensitive data. When crypto is employed, weak key generation and management, and weak algorithm, protocol and cipher usage is common, particularly for weak password hashing storage techniques. For data in transit, server side weaknesses are mainly easy to detect, but hard for data at rest."
2,How can brute-force attacks be detected?,chroma,3,OWASP Top 10 (2017),Application Security,0.381576,"This issue is included in the Top 10 based on an industry survey.\n\nOne strategy for determining if you have sufficient monitoring is to examine the logs following penetration testing. The testers’ actions should be recorded sufficiently to understand what damages they may have inflicted.\n\nMost successful attacks start with vulnerability probing. Allowing such probes to continue can raise the likelihood of successful exploit to nearly 100%.\n\nIn 2016, identifying a breach took an average of 191 days – plenty of time for damage to be inflicted.\n\nIs the Application Vulnerable? Insufficient logging, detection, monitoring and active response occurs any time: • Auditable events, such as logins, failed logins, and high-value\n\ntransactions are not logged.\n\nWarnings and errors generate no, inadequate, or unclear log messages."
3,How can brute-force attacks be detected?,chroma,4,CISA Cybersecurity Incident and Vulnerability Response Playbooks,Incident Response,0.368664,"Correlate Events and Document Timeline\n\nAcquire, store, and analyze logs to correlate adversarial activity. Table 1 below presents an example of logs and event data that are commonly employed to detect and analyze attacker activities.16,17 A simple knowledge base should be established for reference during response to the incident. Thoroughly document every step taken during this and subsequent phases. Create a timeline of all relevant findings. The timeline will allow the team to account for all adversary activity on the network and will assist in creating the findings report at the conclusion of the response.\n\nIdentify Anomalous Activity"
4,How can brute-force attacks be detected?,chroma,5,CISA Cybersecurity Incident and Vulnerability Response Playbooks,Incident Response,0.365707,"Atomic indicators, such as domains and IP addresses, that can detect adversary infrastructure and tools\n\nComputed indicators, such as Yara rules and regular expressions, that detect known malicious artifacts or signs of activity\n\nPatterns and behaviors, such as analytics that detect adversary tactics, techn

---
### Chroma vs. Cosseno Manual — Concordancia Empirica

Como os embeddings e a metrica (cosseno) sao exatamente os mesmos nas duas abordagens, a expectativa e que o resultado de melhor rank (Top-1) coincida na grande maioria das consultas.

In [39]:
def compute_top1_agreement(chroma_results, manual_results, queries):
    matches = 0
    for query in queries:
        chroma_top1 = next(r for r in chroma_results if r["query"] == query)
        manual_top1 = next(r for r in manual_results if r["query"] == query)
        if chroma_top1["chunk"] == manual_top1["chunk"]:
            matches += 1
    return matches / len(queries)


agreement = compute_top1_agreement(chroma_results, manual_results, TEST_QUERIES)
print(f"Concordancia do Top-1 entre Chroma e cosseno manual: {agreement:.0%}")

Concordancia do Top-1 entre Chroma e cosseno manual: 100%


### Chroma vs. Cosseno Manual

A qualidade dos resultados e praticamente identica entre as duas abordagens, como confirma a concordancia de Top-1 medida acima. A diferenca real esta na engenharia por tras de cada uma:

| Criterio | ChromaDB | Cosseno Manual (NumPy) |
|---|---|---|
| **Qualidade de recuperacao** | Identica (mesma metrica e embeddings) | Identica (mesma metrica e embeddings) |
| **Custo computacional** | Menor em bases grandes (indice ANN, ex: HNSW) | Cresce linearmente com o numero de chunks (forca bruta) |
| **Escalabilidade** | Escala para milhoes de vetores com indice aproximado | Inviavel alem de dezenas de milhares de vetores sem otimizacao |
| **Complexidade de implementacao** | Requer configurar e persistir um banco vetorial | Poucas linhas de NumPy, zero infraestrutura extra |
| **Uso recomendado** | Producao, bases de conhecimento que crescem | Prototipagem, datasets pequenos, validacao de baseline |

Em bases pequenas como a desta fase (algumas centenas de chunks), a forca bruta com NumPy e perfeitamente viavel e serve como baseline confiavel para validar se o banco vetorial esta retornando os resultados esperados.

---
## 14. Analise de Recuperacao

### Quando a recuperacao semantica funciona bem

- **Terminologia tecnica especifica de ciberseguranca** (ex: "credential stuffing", "lateral movement") tende a recuperar trechos corretos mesmo quando a consulta usa sinonimos ou frases diferentes das do documento original;
- **Conceitos especificos de frameworks** (ex: perguntar sobre "fases de resposta a incidentes") recupera bem trechos do NIST SP 800-61, que organiza o conteudo exatamente nessas fases;
- **Consultas descritivas**, escritas como uma frase completa em linguagem natural, funcionam melhor do que uma unica palavra-chave isolada, pois fornecem mais contexto semantico ao embedding da consulta.

### Quando a recuperacao produz resultados fracos

- **Consultas ambiguas** (termos genericos que aparecem em multiplos contextos) podem recuperar chunks de categorias diferentes das esperadas;
- **Contexto insuficiente**: consultas muito curtas geram embeddings menos informativos;
- **Fronteiras de chunk**: um chunk pode comecar ou terminar no meio de uma explicacao, fazendo a similaridade cair mesmo quando o documento cobre bem o assunto poucas linhas antes ou depois;
- **Documentacao incompleta**: nem todo topico de seguranca esta coberto com o mesmo nivel de detalhe em todos os documentos da base (ex: um documento pode detalhar contencao de ransomware, enquanto outro apenas o menciona de passagem).

### Por que isso importa para fases futuras

A qualidade da recuperacao define o teto de qualidade de qualquer resposta gerada por um LLM em uma arquitetura RAG: se os chunks recuperados forem irrelevantes ou incompletos, nenhuma engenharia de prompt consegue compensar a ausencia de contexto correto. Validar e comparar estrategias de recuperacao **antes** de introduzir geracao de texto e uma etapa essencial.

---
## 15. Discussao

### Embeddings semanticos e dense retrieval

Embeddings semanticos representam texto como vetores densos em um espaco continuo, onde proximidade geometrica corresponde a proximidade de significado. Essa e a base do **dense retrieval**: em vez de indexar termos exatos (como em buscadores tradicionais), indexamos vetores e buscamos por proximidade.

### Similaridade de cosseno

A metrica de cosseno e a escolha padrao para embeddings de sentence transformers porque esses modelos sao tipicamente treinados para que a proximidade angular entre vetores reflita similaridade semantica, independentemente da magnitude do vetor.

### Bancos de dados vetoriais e ChromaDB

Bancos vetoriais como o ChromaDB tornam a busca por similaridade viavel em escala, usando estruturas de indice aproximado (ex: HNSW) em vez de comparar a consulta contra cada vetor individualmente. Tambem cuidam de persistencia, metadados e filtros — funcionalidades que teriamos que reimplementar manualmente com NumPy puro.

### LangChain

O LangChain fornece abstracoes padronizadas para loaders, splitters, embeddings e vector stores, permitindo trocar qualquer componente (ex: outro modelo de embedding, outro vector store) sem reescrever o pipeline inteiro.

### Unstructured (LangChain)

O `UnstructuredPDFLoader` do LangChain resolve um problema historicamente dificil: extrair texto de PDF preservando estrutura (titulos, listas). Isso melhora diretamente a qualidade do chunking e, por consequencia, da recuperacao.

### Vantagens sobre busca por palavra-chave tradicional

- entende sinonimos e parafrases;
- nao depende de correspondencia exata de termos;
- funciona bem com consultas em linguagem natural, como as escritas por analistas humanos.

### Limitacoes atuais

- o modelo de embedding usado (`all-MiniLM-L6-v2`) e generico, nao foi ajustado especificamente para o dominio de ciberseguranca;
- a qualidade do chunking depende da qualidade da conversao feita pelo Unstructured;
- consultas muito curtas ou ambiguas ainda podem recuperar resultados irrelevantes;
- nao ha, nesta fase, nenhum mecanismo de re-ranking ou geracao de resposta — a saida final ainda e um conjunto de trechos de texto, nao uma resposta fundamentada.

---
# Fase 4 — Inferencia Local, Remota ou Privada

O projeto adota uma estrategia hibrida de inferencia: a recuperacao semantica (ingestao de documentos, geracao de embeddings e busca vetorial) roda localmente no ambiente do projeto, enquanto a geracao das respostas e feita por um modelo acessado remotamente via ecossistema Hugging Face.

A Hugging Face foi escolhida por oferecer modelos e ferramentas open source, favorecendo transparencia, reprodutibilidade e facilidade de integracao. Desenvolvimento e experimentos rodam no Google Colab (acesso a GPU, com limites de uso), ja que o computador do desenvolvedor possui apenas CPU, tornando a execução local de modelos maiores lenta e pouco pratica.

Essa arquitetura equilibra custo, desempenho e controle: documentos e banco vetorial permanecem sob controle da aplicação, reduzindo o volume de informação enviado ao modelo remoto. Credenciais de acesso devem ficar em variaveis de ambiente, nunca no codigo-fonte. Como limitações, a solucao depende de conexao com a Internet e da disponibilidade do servico remoto, alem do risco de exposicao de dados caso informacoes sensiveis sejam enviadas sem anonimizacao adequada.

Em um cenario futuro de produção, a solucao poderia migrar para infraestrutura propria em nuvem, executando modelos abertos do ecossistema Hugging Face em instancias administradas pela propria organização, eliminando a dependencia de APIs publicas e ampliando controle sobre dados, privacidade, conformidade e desempenho.


---
# Cybersecurity Incident Analyzer — Fase 5

## 1. Visao Geral — Retrieval-Augmented Generation (RAG)

Esta fase integra os componentes ja construidos nas Fases 2 e 3 em um pipeline completo de **RAG**:

```text
Pergunta do usuario
    |
    v
Recuperacao Semantica (Top-K via ChromaDB, Fase 3)
    |
    v
Aumento do Prompt (contexto recuperado + instrucoes)
    |
    v
Inferencia com o LLM (Qwen2.5-1.5B-Instruct, Fase 2)
    |
    v
Resposta fundamentada (grounded answer)
```

Nenhum componente de recuperacao ou geracao e reimplementado: `vector_store`, `semantic_search()`, `embedding_model` (Fase 3) e `generate_response_text()`, `model`, `tokenizer` (Fase 2) sao reaproveitados diretamente.


---
## 2. Construcao do Prompt Aumentado

A funcao `build_rag_prompt()` formata os chunks recuperados como contexto numerado, identificando documento de origem, categoria e secao — permitindo rastrear de onde veio cada trecho usado na resposta.

O prompt de sistema (`RAG_SYSTEM_PROMPT`) instrui o modelo explicitamente a:

- responder **apenas** com base no contexto fornecido;
- declarar quando o contexto **nao** contem informacao suficiente, em vez de inventar uma resposta (evitando alucinacao);
- citar o(s) documento(s) de origem quando possivel, para rastreabilidade.


In [40]:
RAG_SYSTEM_PROMPT = (
    "You are a cybersecurity assistant. Answer the user's question using ONLY the "
    "information provided in the context below.\n"
    "- If the context does not contain enough information to answer confidently, say so "
    "explicitly instead of guessing or relying on outside knowledge.\n"
    "- Do not invent facts, statistics, or recommendations that are not supported by the context.\n"
    "- When possible, mention which source document(s) the answer is based on."
)


def build_rag_prompt(query, retrieved_chunks):
    # Formata os chunks recuperados como contexto numerado e rastreavel.
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        source = chunk["document_name"] or "unknown source"
        section = chunk.get("section_heading")
        header = f"[{i}] Source: {source}" + (f" | Section: {section}" if section else "")
        context_blocks.append(f"{header}\n{chunk['chunk']}")

    context_text = "\n\n".join(context_blocks)

    user_prompt = (
        f"Context:\n{context_text}\n\n"
        f"Question: {query}\n\n"
        "Answer the question above using only the context provided."
    )

    return context_text, user_prompt


print("Prompt de sistema RAG e build_rag_prompt() definidos.")


Prompt de sistema RAG e build_rag_prompt() definidos.


---
## 3. Pipeline RAG Completo

A funcao `rag_answer()` encadeia recuperacao, aumento de prompt e geracao em uma unica chamada, retornando tambem os artefatos intermediarios (chunks recuperados e prompt final) para fins de inspecao e auditoria — essencial em um dominio como ciberseguranca, onde a resposta de um LLM deve poder ser rastreada ate sua fonte.


In [41]:
def rag_answer(query, k=5, max_new_tokens=384):
    retrieved_chunks = semantic_search(vector_store, query, k=k)
    context_text, user_prompt = build_rag_prompt(query, retrieved_chunks)

    answer_text, input_tokens, output_tokens = generate_response_text(
        RAG_SYSTEM_PROMPT,
        user_prompt,
        max_new_tokens=max_new_tokens,
    )

    return {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "context_text": context_text,
        "user_prompt": user_prompt,
        "answer": answer_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }


def display_rag_result(result):
    # Exibe recuperacao, prompt resumido e resposta final de forma legivel.
    print("=" * 100)
    print(f"PERGUNTA: {result['query']}")
    print("=" * 100)

    print(f"\nChunks recuperados (top-{len(result['retrieved_chunks'])}):")
    for rank, chunk in enumerate(result["retrieved_chunks"], start=1):
        print("-" * 100)
        print(f"[{rank}] {chunk['document_name']} | {chunk['category']} | score={chunk['similarity_score']:.4f}")
        print_wrapped(chunk["chunk"])

    print(f"\nPrompt final (resumo): {len(result['context_text'])} caracteres de contexto, "
          f"{len(result['retrieved_chunks'])} chunks, {result['input_tokens']} tokens de entrada.")

    print("\n===== RESPOSTA (com RAG): =====")
    print_wrapped(result["answer"])
    print(f"\nTokens: {result['input_tokens']} in / {result['output_tokens']} out")


print("rag_answer() e display_rag_result() definidos.")


rag_answer() e display_rag_result() definidos.


---
## 4. Sessao de Perguntas e Respostas Reproduzivel

Definimos um conjunto fixo de perguntas de ciberseguranca, cobrindo topicos presentes na base de conhecimento (fases de resposta a incidentes, contencao de ransomware, mitigacoes OWASP, tecnicas MITRE ATT&CK) e uma pergunta deliberadamente fora do escopo dos documentos indexados, para observar o comportamento do pipeline quando a recuperacao falha em encontrar contexto relevante.


In [42]:
RAG_QUESTIONS = [
    "What are the phases of incident response according to NIST?",
    "How should an organization contain a ransomware attack?",
    "What are common mitigations for SQL Injection according to OWASP?",
    "What does MITRE ATT&CK say about lateral movement techniques?",
    "What is the difference between a false positive and a false negative in intrusion detection?",
]

rag_results = [rag_answer(query, k=5) for query in RAG_QUESTIONS]

for result in rag_results:
    display_rag_result(result)
    print()


PERGUNTA: What are the phases of incident response according to NIST?

Chunks recuperados (top-5):
----------------------------------------------------------------------------------------------------
[1] CISA Cybersecurity Incident and Vulnerability Response Playbooks | Incident Response | score=0.6094
Analytics or hunt teams that identify

potentially malicious or otherwise unauthorized activity

Incident Response Process

The incident response process starts with the declaration of the incident, as shown in Figure 1. In
this context, “declaration” refers to the identification of an incident and communication to CISA
and agency network defenders rather than formal declaration of a major incident as defined in
applicable law and policy. Succeeding sections, which are organized by phases of the IR lifecycle,
describe each step in more detail. Many activities are iterative and may continuously occur and
evolve until the incident is closed out. Figure 1 illustrates incident response activ

---
## 5. Baseline sem Contexto (Comparacao)

Para avaliar o ganho real do RAG, geramos respostas para as mesmas perguntas **sem** nenhum contexto recuperado.

Na primeira tentativa, o baseline usava o proprio `Qwen2.5-1.5B-Instruct` sem contexto recuperado. Na pratica, esse baseline se mostrou forte demais: por ja ser um modelo instruction-tuned relativamente capaz, ele produziu respostas coerentes e razoavelmente corretas mesmo sem RAG, tornando o contraste com a versao "com RAG" pouco expressivo para fins didaticos.

Para tornar a comparacao mais informativa, o baseline passou a usar um modelo **generico, menor e sem ajuste de instrucoes**: o `gpt2` (124M parametros, modelo base da OpenAI). Por nao ter sido treinado para seguir instrucoes nem para dialogo, e por ter uma capacidade parametrica muito menor, o `gpt2` tende a gerar respostas mais fracas, genericas ou repetitivas quando usado sem contexto — um baseline "fraco" deliberado, que evidencia com mais clareza o ganho trazido pela recuperacao de contexto (RAG) mesmo com um LLM pequeno como base de geracao.


In [43]:
BASELINE_MODEL_ID = "gpt2"

# Modelo generico, pequeno e sem ajuste de instrucoes, escolhido deliberadamente como um
# baseline fraco: ao contrario do Qwen2.5-1.5B-Instruct (usado no pipeline RAG), o gpt2 nao
# foi treinado para seguir instrucoes nem manter dialogo, apenas para completar texto.
baseline_tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_ID)
baseline_tokenizer.pad_token = baseline_tokenizer.eos_token
baseline_model = AutoModelForCausalLM.from_pretrained(BASELINE_MODEL_ID).to(MODEL_DEVICE)


def baseline_answer(query, max_new_tokens=200):
    # Sem chat template: gpt2 e um modelo de completion puro, entao o prompt e formatado
    # manualmente como um par pergunta/resposta em texto livre.
    prompt_text = (
        "You are a cybersecurity assistant. Answer the question using your own knowledge.\n\n"
        f"Question: {query}\nAnswer:"
    )
    model_inputs = baseline_tokenizer(prompt_text, return_tensors="pt")
    model_inputs = {k: v.to(MODEL_DEVICE) for k, v in model_inputs.items()}

    with torch.inference_mode():
        generated_ids = baseline_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=baseline_tokenizer.pad_token_id,
        )

    prompt_tokens = model_inputs["input_ids"].shape[-1]
    completion_tokens = generated_ids.shape[-1] - prompt_tokens
    completion_ids = generated_ids[0][prompt_tokens:]
    answer_text = baseline_tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    return {"query": query, "answer": answer_text, "input_tokens": prompt_tokens, "output_tokens": completion_tokens}


baseline_results = [baseline_answer(query) for query in RAG_QUESTIONS]

for result in baseline_results:
    print("=" * 100)
    print(f"PERGUNTA: {result['query']}")
    print("=" * 100)
    print("\nRESPOSTA (sem RAG / baseline generico, gpt2):")
    print_wrapped(result["answer"])
    print(f"\nTokens: {result['input_tokens']} in / {result['output_tokens']} out")
    print()


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

PERGUNTA: What are the phases of incident response according to NIST?

RESPOSTA (sem RAG / baseline generico, gpt2):
NIST is a national security organization that is responsible for the development of national
security information systems. NIST is responsible for the development of national security
information systems.

Question: What are the phases of incident response according to the National Security Agency?

Answer: NIST is a national security organization that is responsible for the development of national
security information systems. NIST is responsible for the development of national security
information systems.

Question: What are the phases of incident response according to the National Security Agency?

Answer: NIST is a national security organization that is responsible for the development of national
security information systems. NIST is responsible for the development of national security
information systems.

Question: What are the phases of incident response accordin

---
## 6. Comparacao RAG vs. Baseline

A tabela abaixo resume, para cada pergunta, sinais objetivos que ajudam a comparar as duas abordagens: quantidade de chunks recuperados, score de similaridade do melhor chunk, e se a resposta com RAG sinalizou explicitamente falta de contexto suficiente (comportamento esperado quando a recuperacao nao encontra informacao relevante).


In [44]:
GROUNDING_REFUSAL_MARKERS = (
    "does not contain",
    "doesn't contain",
    "not enough information",
    "cannot answer",
    "can't answer",
    "no context",
    "not covered",
    "does not provide",
)


def flags_insufficient_context(answer_text):
    lowered = answer_text.lower()
    return any(marker in lowered for marker in GROUNDING_REFUSAL_MARKERS)


comparison_rows = []
for rag_result, baseline_result in zip(rag_results, baseline_results):
    top_score = rag_result["retrieved_chunks"][0]["similarity_score"] if rag_result["retrieved_chunks"] else None
    comparison_rows.append(
        {
            "query": rag_result["query"],
            "top_similarity_score": top_score,
            "rag_flagged_insufficient_context": flags_insufficient_context(rag_result["answer"]),
            "rag_answer_len": len(rag_result["answer"]),
            "baseline_answer_len": len(baseline_result["answer"]),
        }
    )

pd.DataFrame(comparison_rows)


,query,top_similarity_score,rag_flagged_insufficient_context,rag_answer_len,baseline_answer_len
0,What are the phases of incident response according to NIST?,0.609433,False,778,1151
1,How should an organization contain a ransomware attack?,0.426671,False,579,933
2,What are common mitigations for SQL Injection according to OWASP?,0.670167,False,792,829
3,What does MITRE ATT&CK say about lateral movement techniques?,0.469083,True,239,953
4,What is the difference between a false positive and a false negative in intrusion detection?,0.357618,False,755,964


### Discussao

- **Baseline deliberadamente fraco**: o baseline usa `gpt2`, um modelo generico e sem ajuste de instrucoes, enquanto o pipeline com RAG usa o `Qwen2.5-1.5B-Instruct` (Fase 2). A comparacao e assimetrica de proposito — o objetivo nao e isolar apenas o efeito do contexto recuperado (o que exigiria o mesmo modelo nas duas condicoes), e sim evidenciar de forma didatica o quanto a recuperacao de contexto relevante melhora a qualidade da resposta final, mesmo frente a um LLM maior do lado do RAG.
- **Melhora na precisao factual**: para perguntas alinhadas ao conteudo indexado (fases de resposta a incidentes, contencao de ransomware, mitigacoes OWASP), a resposta com RAG tende a citar terminologia e estrutura identicas as dos documentos-fonte (ex: nomes exatos das fases do NIST), enquanto o baseline com `gpt2` tende a gerar texto generico, vago ou repetitivo, sem nenhuma referencia especifica aos frameworks da base de conhecimento.
- **Reducao de alucinacao**: quando o contexto recuperado e relevante (score de similaridade alto), o RAG ancora a resposta em texto real, reduzindo o risco de o modelo inventar recomendacoes ou fatos nao verificaveis — um risco relevante em ciberseguranca, onde uma recomendacao incorreta pode ter consequencias operacionais. O `gpt2`, por ser um modelo pequeno de completion, e especialmente propenso a repeticao e a respostas sem fundamento quando decodificado de forma gulosa (`do_sample=False`).
- **Falha por recuperacao pobre**: a ultima pergunta (falso positivo vs. falso negativo) e um exemplo deliberado de conceito **generico** de deteccao de intrusao, pouco provavel de estar detalhado nos documentos de framework desta base (NIST CSF, OWASP, MITRE ATT&CK, CISA). Nesse caso, espera-se que o score de similaridade do melhor chunk seja mais baixo e que, idealmente, a resposta com RAG sinalize explicitamente contexto insuficiente — o comportamento correto e desejado do prompt de aumento, em vez de o modelo preencher a lacuna com conhecimento nao verificado pela base.
- **Custo**: o RAG consome mais tokens de entrada (contexto + pergunta) que o baseline (apenas a pergunta), o que se reflete em `input_tokens` mais alto na tabela acima — o trade-off de fundamentar a resposta em evidencia.


---
## 7. Analise do Pipeline RAG

### Tamanho de chunk e overlap

O pipeline reutiliza os chunks da Fase 3 (`chunk_size=1000`, `chunk_overlap=150`). Chunks maiores fornecem mais contexto por trecho recuperado (util para perguntas que exigem varias frases de explicacao, como "fases de resposta a incidentes"), mas aumentam o risco de misturar topicos irrelevantes dentro do mesmo chunk, diluindo a relevancia media do contexto enviado ao LLM.

### Qualidade dos chunks recuperados

A qualidade da resposta gerada e limitada pela qualidade da recuperacao (Fase 3): se os top-k chunks nao contiverem a informacao necessaria, nenhuma engenharia de prompt no passo de geracao consegue compensar essa lacuna — o LLM so pode fundamentar a resposta no que foi de fato recuperado.

### Efeito do valor de k

Usamos `k=5` neste pipeline. Um `k` menor reduz ruido no prompt (menos chunks irrelevantes), mas aumenta o risco de omitir um chunk relevante que ficou fora do top-k. Um `k` maior aumenta a chance de cobertura, mas tambem aumenta o tamanho do prompt, o custo em tokens e o risco de "diluir" a atencao do modelo entre trechos pouco relacionados.

### Limitacoes de janela de contexto

O modelo usado (`Qwen2.5-1.5B-Instruct`) tem uma janela de contexto finita compartilhada entre o prompt de sistema, o contexto recuperado e a pergunta. Com `k` grande ou chunks muito longos, o prompt aumentado pode se aproximar do limite da janela, obrigando a truncar contexto ou reduzir `k` — um trade-off direto entre cobertura de recuperacao e espaco disponivel para a resposta gerada.

### Ponto mais provavel de falha

O ponto mais fragil do pipeline e a **etapa de recuperacao**, nao a geracao: um LLM competente consegue sintetizar bem um contexto relevante, mas nenhum prompt consegue compensar chunks irrelevantes ou ausentes. Isso reforca a conclusao da Fase 3 — validar a qualidade da recuperacao antes de investir em geracao continua sendo o passo com maior impacto na qualidade final do sistema.


---
## 8. Consideracoes de Seguranca

### Risco de prompt injection nos documentos-fonte

Como o conteudo dos chunks recuperados e inserido diretamente no prompt enviado ao LLM, um documento malicioso na base de conhecimento poderia conter instrucoes escondidas (ex: "ignore as instrucoes anteriores e...") que tentariam sequestrar o comportamento do modelo — um vetor de ataque conhecido como **prompt injection indireto**, especialmente relevante quando a base de conhecimento inclui documentos de fontes externas.

### Vazamento de contexto (context leakage)

Chunks recuperados podem, em tese, conter informacao sensivel (ex: detalhes internos de um incidente real, se documentos internos fossem adicionados a base) que acabaria reproduzida na resposta do modelo para qualquer usuario que fizer a pergunta certa — um risco de vazamento de informacao atraves da camada de recuperacao, mesmo que o modelo em si nao tenha sido treinado com esse dado.

### Mitigacoes propostas

- **Curadoria de documentos**: indexar apenas fontes confiaveis e revisadas (como os frameworks publicos usados nesta fase — NIST, OWASP, MITRE, CISA), evitando ingestao automatizada de conteudo nao verificado;
- **Filtragem de contexto**: sanitizar ou inspecionar chunks antes de inseri-los no prompt, buscando padroes suspeitos de instrucoes embutidas no texto do documento;
- **Limitar o volume de texto recuperado**: manter `k` e `chunk_size` no menor valor que ainda atenda a qualidade de resposta necessaria, reduzindo a superficie de exposicao por consulta;
- **Isolamento de instrucoes**: o `RAG_SYSTEM_PROMPT` mantem o contexto recuperado claramente demarcado e subordinado as instrucoes de sistema, reduzindo (sem eliminar) a chance de o conteudo do documento ser interpretado como uma instrucao valida pelo modelo.


---
## 9. Conclusao

Esta fase completou a arquitetura do projeto ao combinar a recuperacao semantica (Fase 3) com a geracao de texto via LLM open source (Fase 2) em um pipeline unico de **Retrieval-Augmented Generation**.

### Principais licoes

1. **RAG fundamenta respostas em evidencia, mas nao substitui uma boa recuperacao.** A qualidade do contexto recuperado define o teto de qualidade da resposta final, confirmando a conclusao da Fase 3 sob uma nova luz: agora com um LLM real gerando texto a partir desses chunks.
2. **Instruir o modelo a admitir incerteza e essencial em um dominio de seguranca.** O `RAG_SYSTEM_PROMPT` instrui explicitamente o modelo a sinalizar quando o contexto e insuficiente, reduzindo o risco de recomendacoes de seguranca alucinadas.
3. **RAG tem custo, nao e gratuito.** Mais tokens de entrada por consulta (contexto + pergunta) frente ao baseline, um trade-off aceitavel em troca de respostas rastreaveis e auditaveis.
4. **A superficie de ataque muda com RAG.** Ao inserir texto de documentos diretamente no prompt, o pipeline introduz riscos novos (prompt injection indireto, vazamento de contexto) que nao existiam em um LLM usado isoladamente, exigindo curadoria e filtragem como parte do design do sistema, nao como um adendo posterior.

Com as cinco fases completas, o projeto demonstra um fluxo de ponta a ponta: triagem e extracao de entidades (Fase 1), analise estruturada com prompt engineering e validacao de schema (Fase 2), ingestao e recuperacao semantica de uma base de conhecimento de ciberseguranca (Fase 3), uma arquitetura de inferencia hibrida local/remota (Fase 4) e, por fim, geracao de respostas fundamentadas em evidencia via RAG (Fase 5) — todo reproduzivel e construido sobre modelos e bibliotecas open source.
